# Rekonsiliasi Hasil Run Lama

Memperbarui keluaran run `00_pipeline_lengkap.ipynb` yang sudah selesai agar sama dengan kode versi
terbaru, **tanpa menjalankan ulang pipeline** (dibangun otomatis oleh `tools/build_merged_notebook.py`,
jangan diedit langsung). Aman dijalankan berulang kali. Berkas yang diubah dicadangkan sekali sebagai
`<nama>.sebelum_rekonsiliasi.<ekstensi>` di folder yang sama.

Perbaikan yang diterapkan:
1. **`age_model.json`.** `deviance_reduction_share` dihapus karena salah (deviance model dengan dan tanpa usia
   dihitung dengan dispersi κ yang berbeda, jadi selisihnya tidak bermakna dan bisa negatif). Diganti
   `loglik_gain_share` = 1 − LL_dengan_usia / LL_tanpa_usia, dihitung dari log-likelihood yang sudah tersimpan.
2. **Pemilihan basis usia** sekarang memakai log-likelihood validasi, bukan deviance (alasan yang sama).
   Notebook ini menghitung ulang pemilihannya. Jika basis terpilih **sama**, target T3 dan semua hasil tetap
   berlaku dan hanya `tables/age_pemilihan_basis.csv` yang diperbarui. Jika **berbeda**, notebook berhenti
   dan menjelaskan tahap yang harus dijalankan ulang.
3. **IK kurva usia** (`tables/age_kurva.csv`). Bootstrap lama mengambil ulang subset sekitar 300 ribu ulasan
   dan memindahkan knot spline di setiap ulangan, sehingga IK tidak berpusat pada kurva yang dilaporkan
   (sebagian titik berada di luar IK-nya sendiri). Sekarang bootstrap memakai seluruh data latih dan knot
   tetap. Kurva (estimasi titik) tidak berubah dan diperiksa sama dengan yang lama. Jika tabel sudah
   dibuat dengan bootstrap baru, langkah ini tidak diulang.
4. **Skor validasi B0** ditambahkan ke `result.json` B0, lalu `tables/validasi_baseline.csv` (hanya B0–B5 dan
   U, tanpa ablasi) dan `tables/validasi_semua.csv` dibuat ulang (sebelumnya B0 tampil NaN).
5. **Urutan acak R0** sebagai batas bawah: `tables/uji_acak.csv` baru, dan kolom R0 ditambahkan ke
   `tables/uji_per_ukuran_produk.csv` (kolom model U diperiksa sama dengan yang lama).
6. **Grafik** `age_kurva.png`, `usia.png` (sumbu y logaritmik), `uji_baseline.png` (dengan R0), dan
   `uji_ukuran_produk.png` (dengan R0) digambar ulang.
7. **Konsistensi per seed** ditambahkan ke `tables/uji_selisih_berpasangan.csv` (kolom `selisih_per_seed`,
   `seed_searah`, `n_pasangan_seed`, `searah_semua_seed`). Selisih gabungan, IK, p-value Holm, dan
   keputusan `lebih_baik` dihitung ulang dan diperiksa sama dengan tabel lama sebelum kolom baru ditulis.

Angka hasil uji yang sudah ada tidak berubah. Langkah 3 butuh GPU (sekitar 10–20 menit, hanya pada
rekonsiliasi pertama).

**Cara pakai di Google Colab:** runtime GPU, isi kategori dan folder Drive yang sama dengan run lama,
lalu *Runtime → Run all*.

In [ ]:
#@title 1 · Siapkan kode (tertanam di notebook ini, tanpa clone repo)
# Kode src/hr dan notebooks/settings.py ditanam di sel ini oleh tools/build_merged_notebook.py
# (versi ebf37aa093df). Di Colab, kode ditulis ke /content/hr_kode setiap sesi. Data TIDAK disimpan di sana.
import os, sys, json, shutil, hashlib, subprocess, tempfile
from pathlib import Path
_KODE = {
    'requirements.txt': 'numpy>=1.26\npandas>=2.1\npyarrow>=14\nscipy>=1.11\nscikit-learn>=1.4\ntorch>=2.2\ntransformers>=4.40\nmatplotlib>=3.8\norjson>=3.9\nnbformat>=5.9\nnbclient>=0.10\nipykernel>=6.29\n',
    'notebooks/settings.py': '"""Satu-satunya tempat mengubah konfigurasi untuk SEMUA notebook.\n\nPenyimpanan\n-----------\n- **Di Google Colab**, Google Drive di-mount otomatis saat sel setup dijalankan. SEMUA data\n  (unduhan mentah ``data/raw``, parquet, embedding, run, tabel, grafik) disimpan di\n  ``HR_DRIVE_ROOT`` (bawaan ``/content/drive/MyDrive/dl-paper``). Setiap notebook Colab berjalan di\n  runtime terpisah, jadi Drive adalah satu-satunya tempat notebook saling bertukar hasil.\n  Jika Drive gagal di-mount, tidak bisa ditulisi, atau terputus di tengah jalan, kode berhenti\n  dengan error, tidak diam-diam menulis ke disk instance.\n- **Di luar Colab**, data disimpan di ``HR_ROOT`` jika diatur, atau di root repo.\n- Cache model Hugging Face (``~/.cache/huggingface``) tetap di instance dan diunduh ulang per sesi.\n\nVariabel lingkungan opsional: ``HR_DRIVE_ROOT`` (folder di Drive), ``HR_MIN_FREE_GB`` (batas ruang\nkosong minimum di Drive, bawaan 10), ``HR_SMOKE=1`` (uji cepat dengan data sintetis, selalu lokal).\n\nPemakaian\n---------\n- Notebook 01-10: ``from settings import cfg`` memakai ``DEFAULTS`` di bawah.\n- ``00_pipeline_lengkap.ipynb``: formulir Colab memanggil ``configure(...)`` dengan nilai pilihan.\n\nPengaturan disimpan di ``outputs/<kategori>/config.json`` saat pertama dipakai. Jika kemudian\npengaturan berubah (selain ``run_priority``) sementara hasil lama masih ada, ``configure`` berhenti\ndengan error agar hasil dari dua pengaturan berbeda tidak tercampur diam-diam.\n"""\nimport importlib.util\nimport json\nimport os\nimport shutil\nimport uuid\nfrom pathlib import Path\n\nfrom hr.config import Config, smoke_config\n\nROOT = Path(__file__).resolve().parents[1]\nSMOKE = os.environ.get("HR_SMOKE") == "1"     # 1 = uji cepat dengan data sintetis dan model mini\ntry:\n    IN_COLAB = importlib.util.find_spec("google.colab") is not None\nexcept ModuleNotFoundError:                   # paket "google" sama sekali tidak terpasang\n    IN_COLAB = False\nDRIVE_MOUNT = "/content/drive"\n\n\ndef connect_drive(drive_root: str | None = None) -> Path:\n    """Mount Google Drive, pastikan folder proyek bisa ditulis dan dibaca, lalu kembalikan path-nya."""\n    from google.colab import drive\n    if not os.path.ismount(DRIVE_MOUNT):\n        drive.mount(DRIVE_MOUNT)\n    if not os.path.ismount(DRIVE_MOUNT):\n        raise RuntimeError(f"Google Drive gagal di-mount di {DRIVE_MOUNT}.")\n    root = Path(drive_root or os.environ.get("HR_DRIVE_ROOT", f"{DRIVE_MOUNT}/MyDrive/dl-paper"))\n    if not str(root.resolve()).startswith(DRIVE_MOUNT + "/"):\n        raise RuntimeError(f"Folder {root} berada di luar Google Drive ({DRIVE_MOUNT}).")\n    if not root.parent.is_dir():\n        raise RuntimeError(f"Folder induk {root.parent} tidak ada di Drive. Periksa folder Drive yang dipilih.")\n    root.mkdir(parents=True, exist_ok=True)\n\n    probe = root / f".tulis_uji_{uuid.uuid4().hex}"\n    token = uuid.uuid4().hex\n    probe.write_text(token)\n    ok = probe.read_text() == token\n    probe.unlink()\n    if not ok:\n        raise RuntimeError(f"Uji tulis-baca di {root} gagal.")\n\n    usage = shutil.disk_usage(root)\n    if usage.total > 0:\n        free_gb, min_gb = usage.free / 1e9, float(os.environ.get("HR_MIN_FREE_GB", "10"))\n        print(f"Google Drive: {free_gb:.1f} GB kosong dari {usage.total / 1e9:.1f} GB")\n        if free_gb < min_gb:\n            raise RuntimeError(\n                f"Ruang kosong Drive {free_gb:.1f} GB < {min_gb:.0f} GB. Unggahan Drive yang melebihi kuota "\n                "bisa gagal tanpa error, jadi kosongkan Drive dulu (atau atur HR_MIN_FREE_GB jika yakin cukup).")\n    else:\n        print("Google Drive: ruang kosong tidak bisa dibaca, pemeriksaan kuota dilewati.")\n    return root\n\n\n# Pengaturan bawaan untuk notebook 01-10. 00_pipeline_lengkap.ipynb mengubahnya lewat formulir Colab.\nDEFAULTS = dict(\n    category="Video_Games",   # KEPUTUSAN KELOMPOK: kategori Amazon Reviews 2023 yang dipakai\n    run_priority=3,           # 1 = wajib saja, 2 = + penting, 3 = semua eksperimen Bab III\n    main_scale="E3",          # skala untuk baseline, ablasi, dan strategi target\n    main_target="T1",         # target perbandingan utama\n)\n_NOT_COMPARED = {"root", "storage_mount", "run_priority"}   # boleh berubah tanpa membatalkan hasil lama\n_cfg: Config | None = None\n\n\ndef _check_saved(c: Config) -> None:\n    """Simpan pengaturan saat pertama dipakai, atau gagal keras jika berbeda dari pengaturan yang\n    menghasilkan keluaran yang sudah ada."""\n    p = c.out_dir / "config.json"\n    if not p.exists():\n        c.save()\n        return\n    old, new = json.loads(p.read_text()), c.as_json()\n    diff = {k: (old.get(k), v) for k, v in new.items() if k not in _NOT_COMPARED and old.get(k) != v}\n    if diff:\n        lines = "\\n".join(f"  {k}: tersimpan={a!r}, sekarang={b!r}" for k, (a, b) in diff.items())\n        raise RuntimeError(\n            f"Pengaturan berbeda dari yang menghasilkan keluaran di {c.out_dir}:\\n{lines}\\n"\n            "Kembalikan pengaturan tersebut, pilih folder Drive lain, atau hapus folder itu untuk mulai dari awal.")\n\n\ndef configure(drive_root: str | None = None, **overrides) -> Config:\n    """Bangun konfigurasi dari DEFAULTS + ``overrides`` (nama field ``Config``) dan sambungkan penyimpanan."""\n    global _cfg\n    if SMOKE:\n        c = smoke_config(ROOT)                     # uji cepat selalu memakai pengaturan mini yang tetap\n    else:\n        unknown = set(overrides) - set(Config.__dataclass_fields__)\n        if unknown:\n            raise TypeError(f"Pengaturan tidak dikenal: {sorted(unknown)}")\n        kw = {**DEFAULTS, **overrides}\n        if IN_COLAB:\n            c = Config(root=connect_drive(drive_root), storage_mount=DRIVE_MOUNT, **kw)\n        else:\n            c = Config(**kw)                       # root = HR_ROOT atau root repo\n    _check_saved(c)\n    _cfg = c\n    print("penyimpanan:", c.root, "(Google Drive)" if c.storage_mount else "(disk lokal)")\n    return c\n\n\ndef configure_saved(category: str, drive_root: str | None = None) -> Config:\n    """Bangun konfigurasi dari ``outputs/<category>/config.json`` run yang sudah ada (dipakai\n    ``00b_rekonsiliasi.ipynb``), sehingga pengaturannya pasti sama dengan yang menghasilkan keluaran."""\n    global _cfg\n    if SMOKE:\n        c = smoke_config(ROOT)\n    else:\n        root = connect_drive(drive_root) if IN_COLAB else Config().root\n        p = Path(root) / "outputs" / category / "config.json"\n        if not p.exists():\n            raise FileNotFoundError(f"{p} tidak ada. Periksa kategori dan folder Drive, atau jalankan pipeline dulu.")\n        saved = json.loads(p.read_text())\n        kw = {k: v for k, v in saved.items() if k not in ("root", "storage_mount")}\n        for k, f in Config.__dataclass_fields__.items():\n            if k in kw and isinstance(f.default, tuple):\n                kw[k] = tuple(kw[k])               # config.json menyimpan tuple sebagai list\n        c = Config(root=Path(root), storage_mount=DRIVE_MOUNT if IN_COLAB else None, **kw)\n    _check_saved(c)                                # tetap gagal keras jika ada field yang tidak cocok\n    _cfg = c\n    print("penyimpanan:", c.root, "(Google Drive)" if c.storage_mount else "(disk lokal)")\n    return c\n\n\ndef __getattr__(name):\n    # ``from settings import cfg`` di notebook 01-10 membangun konfigurasi bawaan saat pertama diminta.\n    if name == "cfg":\n        return _cfg if _cfg is not None else configure()\n    raise AttributeError(name)\n\n\ndef finish() -> None:\n    """Panggil di akhir notebook. Di Colab, memaksa semua tulisan tersinkron ke Drive sebelum runtime\n    ditutup. Setelah ini akses path apa pun akan gagal keras sampai sel pengaturan dijalankan ulang."""\n    if IN_COLAB and not SMOKE and os.path.ismount(DRIVE_MOUNT):\n        from google.colab import drive\n        drive.flush_and_unmount()\n        print("Semua hasil sudah tersinkron ke Google Drive.")\n    else:\n        print("Selesai. Hasil ada di", _cfg.root if _cfg is not None else "(belum dikonfigurasi)")\n',
    'src/hr/__init__.py': '"""Kode eksperimen: Review Helpfulness Ranking dengan embedding BLaIR dan MLP pada Amazon Reviews 2023."""\n',
    'src/hr/config.py': '"""Konfigurasi eksperimen.\n\nSemua nilai yang harus ditetapkan sebelum eksperimen (kategori, seed, proporsi split,\nskala data, grid hiperparameter, ambang relevansi) dikumpulkan di sini agar tidak\ntersebar di notebook. Ubah nilai lewat ``Config(...)`` di sel pertama setiap notebook,\nbukan dengan mengedit kode di tengah notebook.\n"""\nfrom __future__ import annotations\n\nimport json\nimport os\nfrom dataclasses import asdict, dataclass, field\nfrom pathlib import Path\n\nHF_RAW_URL = (\n    "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/"\n    "raw/review_categories/{category}.jsonl"\n)\n\n# Encoder yang dibandingkan. B5 memakai RoBERTa-base karena BLaIR dilatih lanjut dari model ini.\nENCODERS = {\n    "blair": "hyp1231/blair-roberta-base",\n    "roberta": "FacebookAI/roberta-base",\n}\n\nRAW_COLUMNS = [\n    "rating", "title", "text", "asin", "parent_asin", "user_id",\n    "timestamp", "helpful_vote", "verified_purchase",\n]\n\n# Fitur terstruktur (Subbab 3.3 dan 3.7). Usia ulasan dan helpful_vote TIDAK boleh masuk ke sini.\nSTRUCT_FEATURES = [\n    "rating", "verified_purchase", "log_review_length", "log_word_count", "log_title_length",\n]\nFORBIDDEN_FEATURES = {"helpful_vote", "age_days", "log_age", "y_t1", "y_t3", "grade_t1", "grade_t3"}\n\n# Registri eksperimen (Tabel 3.5). "priority" mengikuti usulan di kritik-reviewer.md:\n# 1 = wajib, 2 = penting, 3 = opsional. Atur Config.run_priority untuk memilih yang dijalankan.\nEXPERIMENTS = {\n    "B0": dict(kind="heuristic", priority=1, desc="Urutan berdasarkan panjang isi ulasan"),\n    "B1": dict(kind="sklearn", rep="tfidf", model="linear", loss="ranknet", feats="all", priority=1,\n               desc="TF-IDF + fitur, ranker linear pairwise"),\n    "B2": dict(kind="torch", rep="tfidf_svd", model="mlp", loss="ranknet", feats="all", priority=3,\n               desc="TF-IDF (SVD 768) + fitur, MLP pairwise"),\n    "B3": dict(kind="torch", rep="blair", model="linear", loss="ranknet", feats="all", priority=3,\n               desc="BLaIR + fitur, ranker linear pairwise"),\n    "B4": dict(kind="torch", rep="blair", model="mlp", loss="mse", feats="all", priority=1,\n               desc="BLaIR + fitur, MLP pointwise MSE"),\n    "B5": dict(kind="torch", rep="roberta", model="mlp", loss="ranknet", feats="all", priority=1,\n               desc="RoBERTa-base beku + fitur, MLP pairwise"),\n    "U": dict(kind="torch", rep="blair", model="mlp", loss="ranknet", feats="all", priority=1,\n              desc="Model utama: BLaIR + fitur, MLP RankNet"),\n    "A1": dict(kind="torch", rep="blair", model="mlp", loss="ranknet", feats=[], priority=3,\n               desc="Ablasi: embedding saja"),\n    "A2": dict(kind="torch", rep="blair", model="mlp", loss="ranknet", feats=["rating"], priority=3,\n               desc="Ablasi: + rating"),\n    "A3": dict(kind="torch", rep="blair", model="mlp", loss="ranknet",\n               feats=["rating", "verified_purchase"], priority=3, desc="Ablasi: + verified_purchase"),\n    # A4 = U (seluruh fitur), tidak dilatih ulang.\n}\n# Baris tabel perbandingan baseline (validasi di notebook 05, uji di notebook 07). Ablasi A1-A3 punya tabel sendiri.\nBASELINE_CODES = ["B0", "B1", "B2", "B3", "B4", "B5", "U"]\nSCALE_PRIORITY = {"E1": 2, "E2": 3, "E3": 2, "E4": 3, "E5": 2}\nTARGET_PRIORITY = {"T1": 1, "T2": 3, "T3": 2}\n\n\ndef _default_root() -> Path:\n    env = os.environ.get("HR_ROOT")\n    if env:\n        return Path(env)\n    return Path(__file__).resolve().parents[2]\n\n\n@dataclass\nclass Config:\n    # --- data\n    category: str = "Video_Games"          # KEPUTUSAN KELOMPOK: kategori dipilih sebelum eksperimen\n    root: Path = field(default_factory=_default_root)\n    seed: int = 42\n    product_sample_rate: float = 1.0       # <1.0 untuk mengambil sebagian produk saat membaca file besar\n    min_reviews_per_product: int = 2\n    split_frac: tuple = (0.8, 0.1, 0.1)\n    max_eval_reviews: int = 200_000        # batas ulasan per partisi validasi/uji (diambil per produk utuh)\n    max_train_reviews_per_product: int = 200\n    scales: dict = field(default_factory=lambda: {\n        "E1": 100_000, "E2": 250_000, "E3": 500_000, "E4": 750_000, "E5": 1_000_000})\n    main_scale: str = "E3"                 # skala untuk perbandingan baseline, ablasi, dan target\n    main_target: str = "T1"                # target untuk perbandingan utama (lihat EXPERIMENTS.md)\n    # --- encoder\n    encoders: dict = field(default_factory=lambda: dict(ENCODERS))\n    max_len: int = 512\n    embed_batch: int = 64\n    # --- pasangan\n    max_pairs_per_product: int = 100       # M pada persamaan (3.5)\n    pair_delta: float = 0.0                # delta pada persamaan (3.5)\n    age_window_grid: tuple = (0.25, 0.5, 1.0)   # Delta_a untuk T2, dalam satuan log(1 + hari)\n    # --- pelatihan\n    hidden: tuple = (256, 64)\n    batch_pairs: int = 1024\n    batch_points: int = 1024\n    max_epochs: int = 30\n    patience: int = 3\n    weight_decay: float = 1e-5\n    lr_grid: tuple = (1e-4, 3e-4, 1e-3)\n    dropout_grid: tuple = (0.1, 0.2, 0.3)\n    linear_c_grid: tuple = (0.1, 1.0, 10.0)\n    seeds: tuple = (42, 43, 44)\n    # --- target dan evaluasi\n    map_relevant_grade: int = 2            # ulasan relevan untuk MAP jika g >= 2 (ditetapkan sebelum eksperimen)\n    age_basis_grid: tuple = ("linear", 4, 6, 8)\n    n_folds_age: int = 5\n    age_bootstrap: int = 100\n    eval_bootstrap: int = 1000\n    run_priority: int = 3                  # 1 = wajib saja, 2 = + penting, 3 = semua\n    # --- penyimpanan\n    storage_mount: str | None = None       # mis. "/content/drive". Jika diisi, setiap akses path gagal keras\n                                           # saat mount ini hilang, agar tidak diam-diam menulis ke disk instance\n\n    # ------------------------------------------------------------------ paths\n    def _root(self) -> Path:\n        if self.storage_mount and not os.path.ismount(self.storage_mount):\n            raise RuntimeError(\n                f"{self.storage_mount} tidak ter-mount (Google Drive terputus atau sudah di-unmount). "\n                "Berhenti agar data tidak tertulis ke disk instance yang hilang saat sesi berakhir. "\n                "Restart runtime lalu jalankan ulang sel setup.")\n        return Path(self.root)\n\n    @property\n    def data_dir(self) -> Path:\n        return self._root() / "data"\n\n    @property\n    def raw_path(self) -> Path:\n        return self.data_dir / "raw" / f"{self.category}.jsonl"\n\n    @property\n    def out_dir(self) -> Path:\n        return self._root() / "outputs" / self.category\n\n    def path(self, *parts: str) -> Path:\n        """Path di dalam folder output kategori. Folder induk dibuat otomatis."""\n        p = self.out_dir.joinpath(*parts)\n        p.parent.mkdir(parents=True, exist_ok=True)\n        return p\n\n    def as_json(self) -> dict:\n        """Pengaturan dalam bentuk yang sama persis dengan isi config.json (tuple menjadi list, dst.)."""\n        return json.loads(json.dumps(asdict(self), default=str))\n\n    def save(self) -> Path:\n        p = self.path("config.json")\n        p.write_text(json.dumps(self.as_json(), indent=2))\n        return p\n\n    def experiments_to_run(self) -> list[str]:\n        return [k for k, v in EXPERIMENTS.items() if v["priority"] <= self.run_priority]\n\n    def scales_to_run(self) -> list[str]:\n        return [k for k, p in SCALE_PRIORITY.items() if p <= self.run_priority]\n\n    def targets_to_run(self) -> list[str]:\n        return [k for k, p in TARGET_PRIORITY.items() if p <= self.run_priority]\n\n\ndef smoke_config(root: Path) -> Config:\n    """Konfigurasi uji cepat dengan data sintetis dan model mini. Dipakai jika HR_SMOKE=1.\n    Hasilnya tidak bermakna secara ilmiah, hanya untuk memastikan semua notebook berjalan."""\n    tiny = "hf-internal-testing/tiny-random-RobertaModel"\n    return Config(category="Synthetic", root=Path(root) / "smoke", max_eval_reviews=10_000,\n                  scales={"E1": 1500, "E2": 3000, "E3": 5000, "E4": 6000, "E5": 7000}, main_scale="E2",\n                  encoders={"blair": tiny, "roberta": tiny}, max_len=64, embed_batch=32, max_epochs=3,\n                  patience=2, lr_grid=(1e-3, 3e-3), dropout_grid=(0.1,), linear_c_grid=(1.0,), seeds=(42, 43),\n                  age_basis_grid=("linear", 4), n_folds_age=3, age_bootstrap=3, eval_bootstrap=200,\n                  age_window_grid=(0.5, 1.0))\n',
    'src/hr/data.py': '"""Pengumpulan dan preprocessing data (Subbab 3.2 dan 3.3)."""\nfrom __future__ import annotations\n\nimport hashlib\nimport html\nimport json\nimport os\nimport re\nimport time\nimport urllib.request\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom .config import HF_RAW_URL, RAW_COLUMNS, Config\n\nMS_PER_DAY = 86_400_000\n# Rentang waktu Amazon Reviews 2023 menurut Hou et al. (Tabel 1): Juni 1996 sampai September 2023.\nVALID_MIN = pd.Timestamp("1995-01-01")\nVALID_MAX = pd.Timestamp("2023-12-31")\n\ntry:  # orjson jauh lebih cepat untuk jutaan baris, tetapi opsional\n    import orjson as _json\n\n    def _loads(line: bytes):\n        return _json.loads(line)\nexcept ImportError:  # pragma: no cover\n    def _loads(line: bytes):\n        return json.loads(line)\n\n\n# ---------------------------------------------------------------- 3.2 pengumpulan\ndef download_raw(cfg: Config, chunk: int = 1 << 22) -> dict:\n    """Unduh berkas User Reviews kategori terpilih, lalu catat ukuran dan checksum."""\n    url = HF_RAW_URL.format(category=cfg.category)\n    dest = cfg.raw_path\n    dest.parent.mkdir(parents=True, exist_ok=True)\n    if not dest.exists():\n        tmp = dest.with_suffix(".part")\n        with urllib.request.urlopen(url, timeout=120) as r, open(tmp, "wb") as f:\n            total = int(r.headers.get("Content-Length", 0))\n            done, t0 = 0, time.time()\n            while True:\n                buf = r.read(chunk)\n                if not buf:\n                    break\n                f.write(buf)\n                done += len(buf)\n                if total:\n                    print(f"\\r{done / 1e9:.2f}/{total / 1e9:.2f} GB", end="")\n            f.flush()\n            os.fsync(f.fileno())\n        # Koneksi yang putus bisa berakhir tanpa exception. Jangan pernah menyimpan berkas terpotong.\n        if total and done != total:\n            tmp.unlink()\n            raise IOError(f"Unduhan tidak lengkap: {done} dari {total} byte. Jalankan ulang sel ini.")\n        if tmp.stat().st_size != done:\n            tmp.unlink()\n            raise IOError(f"Ukuran berkas di disk ({tmp.stat().st_size}) tidak sama dengan yang diunduh ({done}).")\n        tmp.rename(dest)\n        print(f"\\nselesai dalam {time.time() - t0:.0f} detik")\n    sha = hashlib.sha256()\n    n_lines = 0\n    with open(dest, "rb") as f:\n        for block in iter(lambda: f.read(1 << 24), b""):\n            sha.update(block)\n            n_lines += block.count(b"\\n")\n    old_path = cfg.path("manifest_raw.json")\n    if old_path.exists():\n        old = json.loads(old_path.read_text())\n        if old.get("sha256") != sha.hexdigest():\n            raise IOError(f"SHA-256 {dest} berbeda dari manifest sebelumnya ({old.get(\'downloaded_at\')}). "\n                          "Berkas berubah atau rusak. Hapus berkas mentah dan manifest lalu unduh ulang.")\n    manifest = {\n        "category": cfg.category,\n        "url": url,\n        "downloaded_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n        "bytes": dest.stat().st_size,\n        "rows": n_lines,\n        "sha256": sha.hexdigest(),\n    }\n    cfg.path("manifest_raw.json").write_text(json.dumps(manifest, indent=2))\n    return manifest\n\n\ndef _keep_product(parent_asin: str, rate: float) -> bool:\n    if rate >= 1.0:\n        return True\n    h = int(hashlib.md5(parent_asin.encode()).hexdigest()[:8], 16) / 0xFFFFFFFF\n    return h < rate\n\n\ndef load_reviews(cfg: Config, limit: int | None = None) -> pd.DataFrame:\n    """Baca JSONL baris demi baris. Pengambilan sampel dilakukan per produk (bukan per baris)\n    supaya daftar ulasan setiap produk tetap utuh."""\n    man_path = cfg.path("manifest_raw.json")\n    if not man_path.exists():\n        raise FileNotFoundError(f"{man_path} tidak ada. Jalankan notebook 01 dulu.")\n    man = json.loads(man_path.read_text())\n    size = cfg.raw_path.stat().st_size\n    if size != man["bytes"]:\n        raise IOError(f"{cfg.raw_path} berukuran {size} byte, manifest mencatat {man[\'bytes\']}. Berkas belum "\n                      "tersinkron penuh ke Drive atau rusak. Tunggu sinkronisasi atau jalankan ulang notebook 01.")\n    cols = {c: [] for c in RAW_COLUMNS}\n    with open(cfg.raw_path, "rb") as f:\n        for i, line in enumerate(f):\n            if limit is not None and i >= limit:\n                break\n            r = _loads(line)\n            pa = r.get("parent_asin") or ""\n            if not _keep_product(pa, cfg.product_sample_rate):\n                continue\n            for c in RAW_COLUMNS:\n                cols[c].append(r.get(c))\n    return pd.DataFrame(cols)\n\n\n# ---------------------------------------------------------------- 3.3 preprocessing\n_TAG = re.compile(r"<[^>]+>")\n_CTRL = re.compile(r"[\\x00-\\x08\\x0b\\x0c\\x0e-\\x1f\\x7f]")\n_WS = re.compile(r"\\s+")\n\n\ndef clean_text(s) -> str:\n    """Rapikan spasi, hapus karakter kendali dan sisa markup HTML. Huruf kapital, tanda baca,\n    angka, dan kata negasi dipertahankan karena dibutuhkan encoder."""\n    if not isinstance(s, str):\n        return ""\n    s = html.unescape(s)\n    s = _TAG.sub(" ", s)\n    s = _CTRL.sub(" ", s)\n    return _WS.sub(" ", s).strip()\n\n\ndef check_timestamp_unit(ts: pd.Series) -> dict:\n    """Langkah 2. Coba satuan milidetik lalu detik, pilih yang rentang tanggalnya masuk akal."""\n    ts = pd.to_numeric(ts, errors="coerce").dropna()\n    out = {}\n    for unit in ("ms", "s"):\n        d = pd.to_datetime(ts, unit=unit, errors="coerce")\n        ok = d.between(VALID_MIN, VALID_MAX).mean()\n        out[unit] = {"min": str(d.min()), "max": str(d.max()), "share_in_range": float(ok)}\n    unit = "ms" if out["ms"]["share_in_range"] >= out["s"]["share_in_range"] else "s"\n    out["chosen"] = unit\n    if out[unit]["share_in_range"] < 0.99:\n        raise ValueError(f"Satuan timestamp tidak meyakinkan: {out}")\n    return out\n\n\ndef preprocess(df: pd.DataFrame, cfg: Config) -> tuple[pd.DataFrame, dict]:\n    """Langkah preprocessing pada Subbab 3.3. Mengembalikan data bersih dan laporan jumlah baris."""\n    rep = {"rows_raw": len(df)}\n    df = df.copy()\n\n    # 1. tipe dan rentang\n    df["rating"] = pd.to_numeric(df["rating"], errors="coerce")\n    df["helpful_vote"] = pd.to_numeric(df["helpful_vote"], errors="coerce")\n    df["verified_purchase"] = df["verified_purchase"].map(\n        {True: 1, False: 0, "true": 1, "false": 0, "True": 1, "False": 0})\n    bad = (~df["rating"].between(1, 5)) | df["helpful_vote"].isna() | (df["helpful_vote"] < 0) \\\n        | df["verified_purchase"].isna()\n    rep["drop_invalid_type"] = int(bad.sum())\n    if rep["drop_invalid_type"] > 0.05 * max(len(df), 1):\n        raise ValueError(f"{rep[\'drop_invalid_type\']} dari {len(df)} baris punya rating/helpful_vote/"\n                         "verified_purchase tidak valid (>5%). Kemungkinan skema berkas berbeda dari yang diharapkan.")\n    df = df[~bad]\n    df["helpful_vote"] = df["helpful_vote"].astype(np.int64)\n    df["verified_purchase"] = df["verified_purchase"].astype(np.int8)\n\n    # 2. satuan timestamp\n    rep["timestamp_check"] = check_timestamp_unit(df["timestamp"])\n    factor = 1 if rep["timestamp_check"]["chosen"] == "ms" else 1000\n    df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce") * factor\n    dt = pd.to_datetime(df["timestamp"], unit="ms", errors="coerce")\n    bad = ~dt.between(VALID_MIN, VALID_MAX)\n    rep["drop_invalid_timestamp"] = int(bad.sum())\n    df = df[~bad]\n    df["timestamp"] = df["timestamp"].astype(np.int64)\n\n    # 3. data kosong\n    df["text"] = df["text"].map(clean_text)       # 4. normalisasi teks\n    df["title"] = df["title"].map(clean_text)\n    empty = df["parent_asin"].isna() | (df["parent_asin"].astype(str).str.len() == 0) | (df["text"].str.len() == 0)\n    rep["drop_empty"] = int(empty.sum())\n    df = df[~empty]\n\n    # 5. duplikasi\n    before = len(df)\n    df = df.drop_duplicates(subset=["user_id", "parent_asin", "timestamp", "text"])\n    rep["drop_duplicate"] = before - len(df)\n\n    # 6. fitur panjang (persamaan 3.2 memakai log(1+x) lalu standardisasi di split.py)\n    df["review_length"] = df["text"].str.len()\n    df["word_count"] = df["text"].str.split().str.len()\n    df["title_length"] = df["title"].str.len()\n    for c in ("review_length", "word_count", "title_length"):\n        df[f"log_{c}"] = np.log1p(df[c].astype(np.float64))\n\n    # usia ulasan (persamaan 3.1). T_ref = timestamp valid terbesar.\n    t_ref = int(df["timestamp"].max())\n    df["age_days"] = (t_ref - df["timestamp"]) / MS_PER_DAY\n    df["log_age"] = np.log1p(df["age_days"])\n    rep["t_ref"] = str(pd.to_datetime(t_ref, unit="ms"))\n\n    # produk dengan sedikitnya dua ulasan\n    sizes = df.groupby("parent_asin")["text"].transform("size")\n    keep = sizes >= cfg.min_reviews_per_product\n    rep["drop_small_product"] = int((~keep).sum())\n    df = df[keep]\n\n    # identitas baris yang stabil dan tidak berkaitan dengan label\n    df = df.sort_values(["parent_asin", "timestamp", "user_id"], kind="stable").reset_index(drop=True)\n    df["rid"] = np.arange(len(df), dtype=np.int64)\n    df["text_hash"] = pd.util.hash_pandas_object(df["text"].str.lower(), index=False).astype(np.uint64)\n    rep["rows_clean"] = len(df)\n    rep["products_clean"] = int(df["parent_asin"].nunique())\n    return df, rep\n\n\ndef describe(df: pd.DataFrame) -> dict:\n    """Statistik deskriptif untuk Bab IV (Subbab 3.2)."""\n    v = df["helpful_vote"]\n    per_prod = df.groupby("parent_asin").size()\n    return {\n        "reviews": len(df),\n        "products": int(per_prod.size),\n        "zero_vote_share": float((v == 0).mean()),\n        "vote_mean": float(v.mean()),\n        "vote_median": float(v.median()),\n        "vote_p99": float(v.quantile(0.99)),\n        "vote_max": int(v.max()),\n        "reviews_per_product_median": float(per_prod.median()),\n        "reviews_per_product_p99": float(per_prod.quantile(0.99)),\n        "age_days_median": float(df["age_days"].median()),\n        "products_all_zero_share": float((df.groupby("parent_asin")["helpful_vote"].max() == 0).mean()),\n        "spearman_age_vote": float(df[["age_days", "helpful_vote"]].rank().corr().iloc[0, 1]),\n    }\n\n\ndef save_parquet(df: pd.DataFrame, path: Path) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    df.to_parquet(path, index=False)\n',
    'src/hr/embed.py': '"""Ekstraksi embedding (Subbab 2.5 dan 3.6) dan representasi TF-IDF (Subbab 2.4).\n\nEmbedding diambil dari token [CLS] (``<s>`` pada RoBERTa) lapisan terakhir lalu dinormalisasi L2,\nsesuai dokumentasi checkpoint BLaIR (persamaan 2.3 dan 2.4). RoBERTa-base untuk B5 memakai\nprosedur yang persis sama.\n"""\nfrom __future__ import annotations\n\nimport json\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\n\ndef _title_for_pair(title: str, max_words: int = 64) -> str:\n    # judul sangat panjang dipotong agar truncation "only_second" selalu bisa memotong isi ulasan\n    w = (title or "").split()\n    return " ".join(w[:max_words])\n\n\ndef token_length_stats(df: pd.DataFrame, model_name: str, max_len: int, n: int = 50_000, seed: int = 0) -> dict:\n    """Proporsi ulasan yang terpotong batas token (Subbab 3.2)."""\n    from transformers import AutoTokenizer\n    tok = AutoTokenizer.from_pretrained(model_name)\n    s = df.sample(min(n, len(df)), random_state=seed)\n    enc = tok([_title_for_pair(t) for t in s["title"]], s["text"].tolist(), truncation=False)\n    lens = np.array([len(x) for x in enc["input_ids"]])\n    return {"sampled": int(len(lens)), "median_tokens": float(np.median(lens)),\n            "p95_tokens": float(np.quantile(lens, 0.95)), "share_truncated": float((lens > max_len).mean())}\n\n\ndef rid_fingerprint(rids) -> str:\n    """Sidik jari urutan baris. Berkas embedding hanya sah untuk urutan rid yang persis sama."""\n    import hashlib\n    return hashlib.sha256(np.ascontiguousarray(np.asarray(rids, dtype=np.int64)).tobytes()).hexdigest()[:16]\n\n\ndef check_embeddings(path: Path, rids, model_name: str | None = None) -> None:\n    """Gagal keras jika berkas embedding tidak lengkap, dibuat dari baris/model lain, atau berisi NaN."""\n    path = Path(path)\n    meta_path = path.with_suffix(".json")\n    if not meta_path.exists():\n        raise FileNotFoundError(f"{meta_path} tidak ada. Embedding {path.name} tidak bisa diverifikasi.")\n    meta = json.loads(meta_path.read_text())\n    E = np.load(path, mmap_mode="r")\n    problems = []\n    if E.shape[0] != len(rids):\n        problems.append(f"jumlah baris {E.shape[0]} != {len(rids)}")\n    if meta.get("fingerprint") != rid_fingerprint(rids):\n        problems.append("urutan rid berbeda (notebook 03 dijalankan ulang?)")\n    if meta.get("done") != E.shape[0]:\n        problems.append(f"ekstraksi belum selesai ({meta.get(\'done\')}/{E.shape[0]})")\n    if model_name is not None and meta.get("model") != model_name:\n        problems.append(f"model {meta.get(\'model\')} != {model_name}")\n    if problems:\n        raise ValueError(f"{path.name} tidak sah: " + "; ".join(problems) +\n                         ". Hapus berkas .npy dan .json-nya lalu jalankan ulang ekstraksi.")\n\n\ndef extract_embeddings(df: pd.DataFrame, model_name: str, out_path: Path, max_len: int = 512,\n                       batch_size: int = 64, device: str | None = None, fp16: bool = True) -> np.memmap:\n    """Hitung embedding sekali tanpa gradien dan simpan sebagai memmap float16.\n\n    Baris ke-k pada berkas sesuai baris ke-k ``df`` (urutkan ``df`` berdasarkan ``rid`` sebelum\n    memanggil). Ekstraksi bisa dilanjutkan jika terputus karena progres disimpan setiap 200 batch.\n    """\n    import torch\n    import torch.nn.functional as F\n    from transformers import AutoModel, AutoTokenizer\n\n    device = device or ("cuda" if torch.cuda.is_available() else "cpu")\n    out_path = Path(out_path)\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n    meta_path = out_path.with_suffix(".json")\n    n = len(df)\n    tok = AutoTokenizer.from_pretrained(model_name)\n    model = AutoModel.from_pretrained(model_name).to(device).eval()\n    dim = model.config.hidden_size                  # 768 untuk BLaIR-RoBERTa-Base dan RoBERTa-base\n    fp = rid_fingerprint(df["rid"])\n    meta = json.loads(meta_path.read_text()) if meta_path.exists() else None\n    resume = out_path.exists() and meta is not None\n    if resume:\n        # Melanjutkan hanya aman jika berkas dibuat dari baris, model, dan pengaturan yang sama persis.\n        same = (meta.get("fingerprint") == fp and meta.get("rows") == n and meta.get("model") == model_name\n                and meta.get("max_len") == max_len)\n        old = np.load(out_path, mmap_mode="r")\n        if not same or old.shape != (n, dim):\n            raise ValueError(f"{out_path.name} sudah ada tetapi dibuat dari data/model/max_len lain "\n                             f"(meta={ {k: meta.get(k) for k in (\'model\', \'rows\', \'max_len\', \'fingerprint\')} }). "\n                             "Hapus berkas .npy dan .json-nya lalu jalankan ulang.")\n        del old\n    else:\n        meta = {"done": 0}\n    mm = np.lib.format.open_memmap(out_path, mode="r+" if resume else "w+", dtype=np.float16, shape=(n, dim))\n    meta.update(fingerprint=fp)\n    if fp16 and device == "cuda":\n        model = model.half()\n\n    titles = [_title_for_pair(t) for t in df["title"].tolist()]\n    texts = df["text"].tolist()\n    order = np.argsort([len(t) for t in texts], kind="stable")  # batch berisi panjang mirip = padding sedikit\n    t0 = time.time()\n    start = meta["done"]\n    with torch.inference_mode():\n        for b, i in enumerate(range(start, n, batch_size)):\n            idx = order[i:i + batch_size]\n            enc = tok([titles[j] for j in idx], [texts[j] for j in idx], truncation="only_second",\n                      max_length=max_len, padding=True, return_tensors="pt").to(device)\n            h = model(**enc).last_hidden_state[:, 0]\n            if not torch.isfinite(h).all():\n                raise FloatingPointError(f"Embedding NaN/inf pada batch mulai baris {i} (overflow fp16?). "\n                                         "Jalankan ulang dengan fp16=False.")\n            h = F.normalize(h.float(), dim=-1)\n            mm[idx] = h.cpu().numpy().astype(np.float16)\n            if b % 200 == 0:\n                mm.flush()\n                meta.update(done=i + len(idx), model=model_name, max_len=max_len, pooling="cls+l2",\n                            rows=n, elapsed_s=time.time() - t0)\n                meta_path.write_text(json.dumps(meta))\n                print(f"\\r{i + len(idx)}/{n} ({(i + len(idx)) / max(time.time() - t0, 1e-9):.0f} ulasan/detik)", end="")\n    mm.flush()\n    # Setiap baris harus sudah ditulis (norma L2 = 1). Baris nol berarti ada batch yang terlewat.\n    for i in range(0, n, 100_000):\n        norms = np.linalg.norm(np.asarray(mm[i:i + 100_000], dtype=np.float32), axis=1)\n        bad = np.flatnonzero(~np.isfinite(norms) | (np.abs(norms - 1) > 0.05))\n        if len(bad):\n            raise ValueError(f"{len(bad)} baris embedding tidak valid mulai baris {i + bad[0]} "\n                             "(belum ditulis atau NaN). Hapus berkas .npy dan .json lalu jalankan ulang.")\n    meta.update(done=n, model=model_name, max_len=max_len, pooling="cls+l2", rows=n,\n                elapsed_s=time.time() - t0, device=device)\n    meta_path.write_text(json.dumps(meta))\n    print("\\nselesai")\n    return mm\n\n\ndef load_embeddings(path: Path) -> np.ndarray:\n    return np.load(path, mmap_mode="r")\n\n\n# ---------------------------------------------------------------- TF-IDF\ndef fit_tfidf(train_texts, max_features: int = 200_000):\n    from sklearn.feature_extraction.text import TfidfVectorizer\n    vec = TfidfVectorizer(min_df=5, max_df=0.9, ngram_range=(1, 2), max_features=max_features,\n                          sublinear_tf=True, dtype=np.float32)\n    vec.fit(train_texts)\n    return vec\n\n\ndef review_text(df: pd.DataFrame) -> list[str]:\n    return (df["title"].fillna("") + " . " + df["text"].fillna("")).tolist()\n\n\ndef fit_svd(X_train, n_components: int = 768, seed: int = 42):\n    """Untuk B2. TF-IDF diperkecil ke 768 dimensi agar sebanding dengan embedding BLaIR."""\n    from sklearn.decomposition import TruncatedSVD\n    svd = TruncatedSVD(n_components=n_components, algorithm="randomized", n_iter=5, random_state=seed)\n    svd.fit(X_train)\n    return svd\n',
    'src/hr/experiment.py': '"""Penghubung antar notebook: memuat data hasil tahap sebelumnya, menjalankan satu konfigurasi\neksperimen, dan menyimpan hasilnya dengan nama yang konsisten.\n\nSetiap run disimpan di ``outputs/<kategori>/runs/<run_id>/`` berisi:\n- ``result.json``   konfigurasi dan metrik validasi terbaik\n- ``history.csv``   loss dan NDCG@10 per epoch (untuk grafik loss)\n- ``pred_val.parquet`` dan ``pred_test.parquet``  skor per ulasan (rid, score)\n- ``model.pt``      bobot model terbaik\n\nMetrik data uji TIDAK dihitung di sini. Notebook 07 menghitungnya sekali setelah semua keputusan\ndiambil.\n"""\nfrom __future__ import annotations\n\nimport json\nimport time\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nfrom .config import EXPERIMENTS, Config\nfrom .embed import check_embeddings\nfrom .features import fuse\nfrom .train import EvalSet, predict, train_scorer\n\n\ndef protocol_cols(target: str) -> tuple[str, str]:\n    """Kolom (grade, target kontinu) untuk evaluasi. T1 dan T2 memakai relevansi mentah."""\n    return ("grade_t3", "y_t3") if target == "T3" else ("grade_t1", "y_t1")\n\n\n@dataclass\nclass Bundle:\n    cfg: Config\n    df: pd.DataFrame                       # semua baris yang punya embedding, sudah berisi target\n    pos: pd.Series                         # rid -> posisi baris pada berkas embedding\n    _emb: dict = field(default_factory=dict)\n\n    def emb(self, rep: str) -> np.ndarray:\n        if rep not in self._emb:\n            path = self.cfg.path(f"emb_{rep}.npy")\n            model = self.cfg.encoders.get(rep)          # None untuk tfidf_svd\n            check_embeddings(path, self.pos.index.to_numpy(), model)\n            self._emb[rep] = np.load(path, mmap_mode="r")\n        return self._emb[rep]\n\n    def rows(self, split: str, scale: str | None = None) -> pd.DataFrame:\n        d = self.df[self.df["split"] == split]\n        if scale is not None:\n            d = d[d[f"in_{scale}"]]\n        return d\n\n    def X(self, rows: pd.DataFrame, rep: str, feats) -> np.ndarray:\n        e = np.asarray(self.emb(rep)[self.pos.loc[rows["rid"]].to_numpy()])\n        return fuse(e, rows, feats)\n\n    def eval_set(self, rows: pd.DataFrame, X: np.ndarray, target: str) -> EvalSet:\n        g, t = protocol_cols(target)\n        return EvalSet(X, rows["parent_asin"].to_numpy(), rows["rid"].to_numpy(),\n                       rows[g].to_numpy(), rows[t].to_numpy())\n\n\ndef load_bundle(cfg: Config) -> Bundle:\n    df = pd.read_parquet(cfg.path("reviews_split.parquet"))\n    tg = pd.read_parquet(cfg.path("targets.parquet"))\n    idx = pd.read_parquet(cfg.path("emb_index.parquet"))\n    df = df.merge(tg, on="rid", how="inner")\n    df = df[df["rid"].isin(idx["rid"])].reset_index(drop=True)\n    pos = pd.Series(np.arange(len(idx)), index=idx["rid"].to_numpy())\n    return Bundle(cfg, df, pos)\n\n\ndef run_id(code: str, scale: str, target: str, seed: int, extra: str = "") -> str:\n    return f"{code}_{scale}_{target}_s{seed}{(\'_\' + extra) if extra else \'\'}"\n\n\ndef run_torch(b: Bundle, code: str, *, scale: str, target: str, seed: int, lr: float, dropout: float,\n              age_window: float | None = None, rep: str | None = None, extra: str = "",\n              overwrite: bool = False, verbose: bool = True) -> dict:\n    """Jalankan satu eksperimen torch (B2-B5, U, A1-A3, E*, T*)."""\n    cfg = b.cfg\n    spec = EXPERIMENTS[code]\n    rep = rep or spec["rep"]\n    rid_ = run_id(code, scale, target, seed, extra)\n    out = cfg.out_dir / "runs" / rid_\n    tr, va, te = b.rows("train", scale), b.rows("val"), b.rows("test")\n    if (out / "result.json").exists() and not overwrite:\n        old = json.loads((out / "result.json").read_text())\n        want = {"rep": rep, "feats": spec["feats"], "model": spec["model"], "loss": spec["loss"], "lr": lr,\n                "dropout": dropout, "age_window": age_window, "train_reviews": len(tr)}\n        diff = {k: (old.get(k), v) for k, v in want.items() if old.get(k) != v}\n        if diff:\n            # Nama run tidak memuat lr/dropout. Tanpa pemeriksaan ini, run lama dengan hiperparameter\n            # atau data berbeda akan dipakai ulang diam-diam.\n            raise ValueError(f"Run {rid_} sudah ada dengan pengaturan berbeda (lama, baru): {diff}. "\n                             "Hapus folder run tersebut atau panggil dengan overwrite=True.")\n        return old\n    out.mkdir(parents=True, exist_ok=True)\n\n    y_col = "y_t3" if target == "T3" else "y_t1"\n    Xtr, Xva, Xte = (b.X(d, rep, spec["feats"]) for d in (tr, va, te))\n    hidden = cfg.hidden if spec["model"] == "mlp" else ()\n    t0 = time.time()\n    model, hist, best = train_scorer(\n        Xtr, tr[y_col].to_numpy(), tr["parent_asin"].to_numpy(), b.eval_set(va, Xva, target),\n        loss=spec["loss"], hidden=hidden, dropout=dropout, lr=lr, weight_decay=cfg.weight_decay,\n        max_epochs=cfg.max_epochs, patience=cfg.patience, batch_pairs=cfg.batch_pairs,\n        batch_points=cfg.batch_points, max_pairs=cfg.max_pairs_per_product, delta=cfg.pair_delta,\n        log_age_train=tr["log_age"].to_numpy() if target == "T2" else None,\n        age_window=age_window if target == "T2" else None, seed=seed, map_rel=cfg.map_relevant_grade,\n        verbose=verbose)\n    device = next(model.parameters()).device.type\n    hist.to_csv(out / "history.csv", index=False)\n    for name, d, X in (("val", va, Xva), ("test", te, Xte)):\n        pd.DataFrame({"rid": d["rid"].to_numpy(), "score": predict(model, X, device)}).to_parquet(\n            out / f"pred_{name}.parquet", index=False)\n    torch.save(model.state_dict(), out / "model.pt")\n    res = {"run_id": rid_, "code": code, "rep": rep, "feats": spec["feats"], "model": spec["model"],\n           "loss": spec["loss"], "scale": scale, "target": target, "seed": seed, "lr": lr, "dropout": dropout,\n           "age_window": age_window, "train_reviews": len(tr), "train_products": int(tr["parent_asin"].nunique()),\n           "best_val_ndcg@10": best, "epochs_run": int(len(hist)),\n           "best_epoch": int(hist.loc[hist["val_ndcg@10"].idxmax(), "epoch"]),\n           "minutes": (time.time() - t0) / 60}\n    (out / "result.json").write_text(json.dumps(res, indent=2, default=str))\n    return res\n\n\ndef save_scores(cfg: Config, run: str, meta: dict, val_rows, val_scores, test_rows, test_scores) -> None:\n    """Simpan skor untuk baseline non-torch (B0, B1) dalam format yang sama. ``meta`` wajib berisi\n    ``best_val_ndcg@10``, karena tabel validasi notebook 05-06 dibaca dari result.json."""\n    v = meta.get("best_val_ndcg@10")\n    if v is None or not np.isfinite(v):\n        raise ValueError(f"{run}: meta harus berisi best_val_ndcg@10 yang valid, bukan {v!r}")\n    out = cfg.out_dir / "runs" / run\n    out.mkdir(parents=True, exist_ok=True)\n    pd.DataFrame({"rid": val_rows["rid"].to_numpy(), "score": val_scores}).to_parquet(out / "pred_val.parquet", index=False)\n    pd.DataFrame({"rid": test_rows["rid"].to_numpy(), "score": test_scores}).to_parquet(out / "pred_test.parquet", index=False)\n    (out / "result.json").write_text(json.dumps({"run_id": run, **meta}, indent=2, default=str))\n\n\ndef results_table(cfg: Config) -> pd.DataFrame:\n    rows = [json.loads(p.read_text()) for p in sorted((cfg.out_dir / "runs").glob("*/result.json"))]\n    return pd.DataFrame(rows)\n\n\ndef validation_table(cfg: Config, scale: str | None = None, target: str | None = None,\n                     codes: list[str] | None = None) -> pd.DataFrame:\n    """Rata-rata, simpangan baku, dan jumlah seed ``best_val_ndcg@10`` per konfigurasi, tanpa run\n    penalaan. Dengan ``scale`` dan ``target``: satu baris per kode (tabel notebook 05, dengan\n    ``codes`` = ``BASELINE_CODES`` agar isinya tidak bergantung pada apakah ablasi sudah dijalankan).\n    Tanpa: per (code, scale, target) (tabel notebook 06). Gagal keras jika ada run tanpa skor\n    validasi, agar baseline tidak diam-diam tampil sebagai NaN."""\n    res = results_table(cfg)\n    res = res[~res["run_id"].str.contains("_tune")]\n    if scale is not None:\n        res = res[(res["scale"] == scale) & (res["target"] == target)]\n    if codes is not None:\n        res = res[res["code"].isin(codes)]\n    bad = res.loc[~np.isfinite(pd.to_numeric(res.get("best_val_ndcg@10"), errors="coerce")), "run_id"].tolist()\n    if bad:\n        raise ValueError(f"Run tanpa best_val_ndcg@10: {bad}. Untuk run lama, jalankan notebooks/00b_rekonsiliasi.ipynb.")\n    if scale is not None:\n        return (res.groupby("code")["best_val_ndcg@10"].agg(["mean", "std", "count"])\n                .sort_values("mean", ascending=False))\n    return res.groupby(["code", "scale", "target"])["best_val_ndcg@10"].agg(["mean", "std", "count"]).reset_index()\n\n\ndef load_pred(cfg: Config, run: str, split: str) -> pd.DataFrame:\n    return pd.read_parquet(cfg.out_dir / "runs" / run / f"pred_{split}.parquet")\n',
    'src/hr/features.py': '"""Feature fusion (persamaan 3.4) dan baseline non-neural (B0, B1)."""\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\nimport scipy.sparse as sp\n\nfrom .config import FORBIDDEN_FEATURES, STRUCT_FEATURES\nfrom .pairs import PairSampler\n\n\ndef struct_matrix(df: pd.DataFrame, feats) -> np.ndarray:\n    """Fitur terstruktur yang sudah distandardisasi (kolom z_*)."""\n    feats = STRUCT_FEATURES if feats == "all" else list(feats)\n    bad = set(feats) & FORBIDDEN_FEATURES\n    assert not bad, f"Fitur terlarang masuk ke model: {bad}"\n    if not feats:\n        return np.zeros((len(df), 0), dtype=np.float16)\n    return df[[f"z_{c}" for c in feats]].to_numpy(np.float16)\n\n\ndef fuse(emb: np.ndarray, df: pd.DataFrame, feats) -> np.ndarray:\n    """x_i = [e_i ; z_i]. ``emb`` harus sejajar baris dengan ``df``."""\n    assert len(emb) == len(df)\n    return np.hstack([np.asarray(emb, dtype=np.float16), struct_matrix(df, feats)])\n\n\n# ---------------------------------------------------------------- B0\ndef score_b0(df: pd.DataFrame) -> np.ndarray:\n    """B0: ulasan diurutkan berdasarkan panjang isi (jumlah karakter)."""\n    return df["review_length"].to_numpy(np.float64)\n\n\n# ---------------------------------------------------------------- B1\ndef train_b1(X_train: sp.csr_matrix, y: np.ndarray, pid: np.ndarray, C: float, max_pairs: int,\n             delta: float, seed: int):\n    """Ranker linear pairwise pada TF-IDF + fitur.\n\n    Regresi logistik tanpa intersep pada selisih vektor x_i - x_j adalah RankNet dengan pemberi\n    skor linear s = w^T x (persamaan 2.12-2.14). Setengah pasangan dibalik agar kelasnya seimbang.\n    """\n    from sklearn.linear_model import LogisticRegression\n    hi, lo = PairSampler(pid, y, max_pairs, delta, seed=seed).sample()\n    rng = np.random.default_rng(seed)\n    flip = rng.random(len(hi)) < 0.5\n    a = np.where(flip, lo, hi)\n    b = np.where(flip, hi, lo)\n    D = X_train[a] - X_train[b]\n    lab = (~flip).astype(np.int8)\n    clf = LogisticRegression(C=C, fit_intercept=False, solver="liblinear", max_iter=200)\n    clf.fit(D, lab)\n    return clf\n\n\ndef score_linear(clf, X) -> np.ndarray:\n    return np.asarray(X @ clf.coef_.ravel()).ravel()\n\n\ndef tfidf_plus_struct(vec, texts, df: pd.DataFrame) -> sp.csr_matrix:\n    return sp.hstack([vec.transform(texts), sp.csr_matrix(struct_matrix(df, "all").astype(np.float32))]).tocsr()\n',
    'src/hr/finetune.py': '"""Fine-tuning encoder dengan objektif pemeringkatan (eksperimen opsional, kritik-reviewer.md 3.3).\n\nEncoder (BLaIR atau RoBERTa-base) tidak dibekukan dan dilatih bersama kepala MLP dengan loss\nRankNet. Setiap langkah mengambil beberapa produk, lalu beberapa ulasan dari setiap produk, dan\nloss dihitung pada semua pasangan di dalam produk dengan y_i - y_j > delta.\n"""\nfrom __future__ import annotations\n\nimport copy\nimport time\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom .embed import _title_for_pair\nfrom .metrics import macro, per_product_metrics\n\n\nclass EncoderRanker(nn.Module):\n    def __init__(self, model_name: str, n_struct: int, hidden=(256, 64), dropout: float = 0.2):\n        super().__init__()\n        from transformers import AutoModel\n        self.encoder = AutoModel.from_pretrained(model_name)\n        d = self.encoder.config.hidden_size + n_struct\n        layers = []\n        for h in hidden:\n            layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]\n            d = h\n        layers.append(nn.Linear(d, 1))\n        self.head = nn.Sequential(*layers)\n\n    def forward(self, enc: dict, z: torch.Tensor) -> torch.Tensor:\n        h = self.encoder(**enc).last_hidden_state[:, 0]\n        h = F.normalize(h, dim=-1)                       # sama dengan versi beku: [CLS] + L2\n        return self.head(torch.cat([h, z], dim=-1).float()).squeeze(-1)\n\n\ndef _within_product_ranknet(s: torch.Tensor, y: torch.Tensor, grp: torch.Tensor, delta: float):\n    same = grp[:, None] == grp[None, :]\n    better = (y[:, None] - y[None, :]) > delta\n    mask = same & better\n    if mask.sum() == 0:\n        return None\n    diff = s[:, None] - s[None, :]\n    return F.softplus(-diff[mask]).mean()\n\n\nclass ProductBatcher:\n    def __init__(self, df: pd.DataFrame, y: np.ndarray, products_per_step: int, reviews_per_product: int,\n                 seed: int):\n        self.rng = np.random.default_rng(seed)\n        self.groups = [np.asarray(ix) for ix in df.groupby("parent_asin").indices.values() if len(ix) >= 2]\n        self.y = y\n        self.P, self.R = products_per_step, reviews_per_product\n\n    def __iter__(self):\n        order = self.rng.permutation(len(self.groups))\n        for i in range(0, len(order), self.P):\n            idx, grp = [], []\n            for k, g in enumerate(order[i:i + self.P]):\n                rows = self.groups[g]\n                pick = self.rng.choice(rows, size=min(self.R, len(rows)), replace=False)\n                idx.append(pick)\n                grp.append(np.full(len(pick), k))\n            yield np.concatenate(idx), np.concatenate(grp)\n\n\n@torch.no_grad()\ndef score_frame(model: EncoderRanker, tok, df: pd.DataFrame, Z: np.ndarray, max_len: int, device: str,\n                batch: int = 128) -> np.ndarray:\n    model.eval()\n    titles = [_title_for_pair(t) for t in df["title"]]\n    texts = df["text"].tolist()\n    order = np.argsort([len(t) for t in texts])\n    out = np.zeros(len(df), dtype=np.float32)\n    for i in range(0, len(df), batch):\n        ix = order[i:i + batch]\n        enc = tok([titles[j] for j in ix], [texts[j] for j in ix], truncation="only_second",\n                  max_length=max_len, padding=True, return_tensors="pt").to(device)\n        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):\n            s = model(enc, torch.as_tensor(Z[ix], device=device, dtype=torch.float32))\n        out[ix] = s.float().cpu().numpy()\n    return out\n\n\ndef finetune(model_name: str, train_df: pd.DataFrame, y_train: np.ndarray, Z_train: np.ndarray,\n             val_df: pd.DataFrame, Z_val: np.ndarray, val_grade: np.ndarray, val_target: np.ndarray, *,\n             max_len: int = 256, epochs: int = 2, products_per_step: int = 16, reviews_per_product: int = 8,\n             lr_encoder: float = 2e-5, lr_head: float = 1e-3, weight_decay: float = 0.01,\n             grad_accum: int = 2, delta: float = 0.0, eval_every: int = 1000, seed: int = 42,\n             device: str | None = None, map_rel: int = 2):\n    from transformers import AutoTokenizer, get_linear_schedule_with_warmup\n    device = device or ("cuda" if torch.cuda.is_available() else "cpu")\n    torch.manual_seed(seed)\n    tok = AutoTokenizer.from_pretrained(model_name)\n    model = EncoderRanker(model_name, Z_train.shape[1]).to(device)\n    opt = torch.optim.AdamW([\n        {"params": model.encoder.parameters(), "lr": lr_encoder},\n        {"params": model.head.parameters(), "lr": lr_head},\n    ], weight_decay=weight_decay)\n    batcher = ProductBatcher(train_df.reset_index(drop=True), y_train, products_per_step, reviews_per_product, seed)\n    steps_per_epoch = int(np.ceil(len(batcher.groups) / products_per_step))\n    total = max(1, epochs * steps_per_epoch // grad_accum)\n    sched = get_linear_schedule_with_warmup(opt, int(0.06 * total), total)\n    scaler = torch.amp.GradScaler("cuda", enabled=device == "cuda")\n    titles = [_title_for_pair(t) for t in train_df["title"]]\n    texts = train_df["text"].tolist()\n\n    def run_eval():\n        s = score_frame(model, tok, val_df, Z_val, max_len, device)\n        pp = per_product_metrics(val_df["parent_asin"].to_numpy(), s, val_grade, val_target,\n                                 val_df["rid"].to_numpy(), map_rel=map_rel)\n        return macro(pp)\n\n    hist, best, best_state, step = [], -np.inf, None, 0\n    t0 = time.time()\n    for ep in range(epochs):\n        model.train()\n        running = []\n        for k, (idx, grp) in enumerate(batcher):\n            enc = tok([titles[j] for j in idx], [texts[j] for j in idx], truncation="only_second",\n                      max_length=max_len, padding=True, return_tensors="pt").to(device)\n            z = torch.as_tensor(Z_train[idx], device=device, dtype=torch.float32)\n            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):\n                s = model(enc, z)\n            loss = _within_product_ranknet(s.float(), torch.as_tensor(y_train[idx], device=device),\n                                           torch.as_tensor(grp, device=device), delta)\n            if loss is None:\n                continue\n            scaler.scale(loss / grad_accum).backward()\n            running.append(loss.item())\n            if (k + 1) % grad_accum == 0:\n                scaler.unscale_(opt)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n                scaler.step(opt)\n                scaler.update()\n                opt.zero_grad()\n                sched.step()\n                step += 1\n                if step % eval_every == 0:\n                    m = run_eval()\n                    hist.append({"epoch": ep + 1, "step": step, "train_loss": float(np.mean(running)),\n                                 "val_ndcg@10": m["ndcg@10"], "minutes": (time.time() - t0) / 60})\n                    print(hist[-1])\n                    running = []\n                    if m["ndcg@10"] > best:\n                        best, best_state = m["ndcg@10"], copy.deepcopy(model.state_dict())\n                    model.train()\n        m = run_eval()\n        hist.append({"epoch": ep + 1, "step": step, "train_loss": float(np.mean(running)) if running else np.nan,\n                     "val_ndcg@10": m["ndcg@10"], "minutes": (time.time() - t0) / 60})\n        print(hist[-1])\n        if m["ndcg@10"] > best:\n            best, best_state = m["ndcg@10"], copy.deepcopy(model.state_dict())\n    model.load_state_dict(best_state)\n    return model, tok, pd.DataFrame(hist), best\n',
    'src/hr/human.py': '"""Validasi label dengan penilaian manusia (eksperimen opsional, kritik-reviewer.md 3.2)."""\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nRUBRIC = """Nilai setiap ulasan dengan skala 1 sampai 5 untuk pertanyaan\n"Seberapa membantu ulasan ini bagi calon pembeli untuk memutuskan membeli produk tersebut?"\n1 = tidak membantu sama sekali, 2 = sedikit membantu, 3 = cukup membantu, 4 = membantu, 5 = sangat membantu.\nNilai setiap ulasan secara mandiri. Jangan berdiskusi dengan penilai lain sebelum semua selesai.\nJumlah suara, tanggal, dan skor model sengaja tidak ditampilkan."""\n\n\ndef make_sheet(test_df: pd.DataFrame, n_products: int = 80, per_product: int = 6, min_reviews: int = 4,\n               seed: int = 7) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Ambil produk dari data uji secara bertingkat menurut ukuran produk, lalu beberapa ulasan per produk.\n\n    Mengembalikan (lembar anotasi tanpa label, kunci rahasia item_id -> rid)."""\n    rng = np.random.default_rng(seed)\n    sizes = test_df.groupby("parent_asin").size()\n    sizes = sizes[sizes >= min_reviews]\n    strata = pd.qcut(sizes.rank(method="first"), 4, labels=False)\n    chosen = []\n    for s in range(4):\n        pool = strata[strata == s].index.to_numpy()\n        chosen += list(rng.choice(pool, size=min(len(pool), n_products // 4), replace=False))\n    rows = []\n    for p_code, p in enumerate(chosen):\n        g = test_df[test_df["parent_asin"] == p]\n        pick = g.sample(min(per_product, len(g)), random_state=int(rng.integers(1 << 31)))\n        for _, r in pick.iterrows():\n            rows.append({"product_code": f"P{p_code:03d}", "rid": int(r["rid"]),\n                         "title": r["title"], "text": r["text"]})\n    sheet = pd.DataFrame(rows).sample(frac=1.0, random_state=seed).reset_index(drop=True)\n    sheet = sheet.sort_values("product_code", kind="stable").reset_index(drop=True)\n    sheet["item_id"] = [f"I{i:04d}" for i in range(len(sheet))]\n    key = sheet[["item_id", "rid", "product_code"]].copy()\n    blind = sheet[["item_id", "product_code", "title", "text"]].copy()\n    blind["score_1_5"] = ""\n    return blind, key\n\n\ndef weighted_kappa(a, b, k: int = 5) -> float:\n    """Cohen\'s kappa berbobot kuadratik untuk dua penilai pada skala 1..k."""\n    a = np.asarray(a, dtype=int) - 1\n    b = np.asarray(b, dtype=int) - 1\n    O = np.zeros((k, k))\n    for i, j in zip(a, b):\n        O[i, j] += 1\n    W = np.array([[(i - j) ** 2 for j in range(k)] for i in range(k)]) / (k - 1) ** 2\n    E = np.outer(O.sum(1), O.sum(0)) / O.sum()\n    return float(1 - (W * O).sum() / (W * E).sum())\n\n\ndef krippendorff_alpha_interval(ratings: np.ndarray) -> float:\n    """Alpha Krippendorff (metrik interval). ``ratings`` berukuran (penilai, item), NaN = kosong."""\n    R = np.asarray(ratings, dtype=np.float64)\n    units = [R[:, j][~np.isnan(R[:, j])] for j in range(R.shape[1])]\n    units = [u for u in units if len(u) >= 2]\n    n = sum(len(u) for u in units)\n    if n == 0:\n        return float("nan")\n    Do_num = sum(((u[:, None] - u[None, :]) ** 2).sum() / (len(u) - 1) for u in units)\n    allv = np.concatenate(units)\n    De = ((allv[:, None] - allv[None, :]) ** 2).sum() / (n * (n - 1))\n    Do = Do_num / n\n    return float(1 - Do / De) if De > 0 else float("nan")\n',
    'src/hr/metrics.py': '"""Metrik pemeringkatan per produk (Subbab 2.10, 3.8, dan 3.9).\n\nAturan yang ditetapkan sebelum eksperimen:\n- NDCG@k (persamaan 2.19-2.20) dihitung pada produk dengan IDCG@k > 0.\n- Spearman (persamaan 2.21) memakai peringkat rata-rata, untuk produk dengan >= 3 ulasan dan\n  target tidak konstan.\n- AP/MAP (persamaan 2.22-2.23) dengan relevan jika g >= ``map_rel``, untuk produk yang punya\n  minimal satu ulasan relevan dan satu tidak relevan.\n- Skor kembar diurutkan secara tetap berdasarkan hash identitas baris yang tidak berkaitan dengan label.\n"""\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.stats import rankdata\n\n\ndef tiebreak_key(rid: np.ndarray) -> np.ndarray:\n    return (np.asarray(rid, dtype=np.uint64) * np.uint64(2654435761)) % np.uint64(2 ** 32)\n\n\ndef dcg(grades_in_order: np.ndarray, k: int) -> float:\n    g = grades_in_order[:k].astype(np.float64)\n    return float(np.sum((2.0 ** g - 1.0) / np.log2(np.arange(2, len(g) + 2))))\n\n\ndef per_product_metrics(pid, score, grade, target, rid, ks=(5, 10), map_rel: int = 2) -> pd.DataFrame:\n    """Hitung semua metrik untuk setiap produk. Nilai NaN berarti produk tidak memenuhi syarat\n    metrik tersebut."""\n    pid = np.asarray(pid)\n    score = np.asarray(score, dtype=np.float64)\n    grade = np.asarray(grade)\n    target = np.asarray(target, dtype=np.float64)\n    tb = tiebreak_key(rid)\n    srt = np.argsort(pid, kind="stable")\n    pid_s = pid[srt]\n    bounds = np.flatnonzero(np.r_[True, pid_s[1:] != pid_s[:-1], True])\n    rows = []\n    for a, b in zip(bounds[:-1], bounds[1:]):\n        idx = srt[a:b]\n        n = len(idx)\n        if n < 2:\n            continue\n        # urutan prediksi: skor menurun, kembar diputus dengan tiebreak_key\n        o = idx[np.lexsort((tb[idx], -score[idx]))]\n        g_pred = grade[o]\n        g_ideal = np.sort(grade[idx])[::-1]\n        r = {"parent_asin": pid_s[a], "n": n}\n        for k in ks:\n            idcg = dcg(g_ideal, k)\n            r[f"ndcg@{k}"] = dcg(g_pred, k) / idcg if idcg > 0 else np.nan\n        t = target[idx]\n        if n >= 3 and np.ptp(t) > 0:\n            rs, rt = rankdata(score[idx]), rankdata(t)\n            r["spearman"] = float(np.corrcoef(rs, rt)[0, 1]) if np.ptp(rs) > 0 else 0.0\n        else:\n            r["spearman"] = np.nan\n        rel = (g_pred >= map_rel).astype(np.float64)\n        R = rel.sum()\n        if 0 < R < n:\n            prec = np.cumsum(rel) / np.arange(1, n + 1)\n            r["ap"] = float(np.sum(prec * rel) / R)\n        else:\n            r["ap"] = np.nan\n        rows.append(r)\n    return pd.DataFrame(rows)\n\n\ndef macro(df: pd.DataFrame, metrics=("ndcg@10", "ndcg@5", "spearman", "ap")) -> dict:\n    """Rata-rata makro (persamaan 3.8) beserta jumlah produk valid per metrik."""\n    out = {}\n    for m in metrics:\n        v = df[m].dropna()\n        name = "map" if m == "ap" else m\n        out[name] = float(v.mean()) if len(v) else float("nan")\n        out[f"n_{name}"] = int(len(v))\n    return out\n\n\ndef coverage(df_eval_rows: pd.DataFrame, per_prod: pd.DataFrame) -> dict:\n    """Berapa produk dan ulasan yang benar-benar tercakup NDCG (kritik reviewer nomor 4)."""\n    sizes = df_eval_rows.groupby("parent_asin").size()\n    valid = set(per_prod.loc[per_prod["ndcg@10"].notna(), "parent_asin"])\n    return {\n        "products_total": int(len(sizes)),\n        "products_ndcg_valid": len(valid),\n        "products_excluded_all_zero": int(len(sizes) - len(valid)),\n        "reviews_total": int(sizes.sum()),\n        "reviews_in_valid_products": int(sizes[sizes.index.isin(valid)].sum()),\n    }\n',
    'src/hr/models.py': '"""Pemberi skor (Subbab 2.6 dan 3.7) dan fungsi loss (Subbab 2.7)."""\nfrom __future__ import annotations\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\nclass Scorer(nn.Module):\n    """MLP pada persamaan (2.5)-(2.7). ``hidden=()`` menghasilkan ranker linear (B3)."""\n\n    def __init__(self, d_in: int, hidden=(256, 64), dropout: float = 0.2):\n        super().__init__()\n        layers, prev = [], d_in\n        for h in hidden:\n            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]\n            prev = h\n        layers.append(nn.Linear(prev, 1))      # lapisan keluaran tanpa aktivasi\n        self.net = nn.Sequential(*layers)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return self.net(x.float()).squeeze(-1)\n\n\n# ---------------------------------------------------------------- loss\ndef ranknet_loss(s_hi: torch.Tensor, s_lo: torch.Tensor) -> torch.Tensor:\n    """Persamaan (2.14) dan (3.6): ln(1 + exp(-(s_i - s_j))) untuk pasangan dengan y_i > y_j.\n    softplus(-x) adalah bentuk yang stabil secara numerik dari ln(1 + exp(-x))."""\n    return F.softplus(-(s_hi - s_lo)).mean()\n\n\ndef ranknet_loss_general(s_i: torch.Tensor, s_j: torch.Tensor, p_target: torch.Tensor) -> torch.Tensor:\n    """Persamaan (2.13), cross-entropy dengan peluang target sembarang. Dipakai untuk pengujian."""\n    return F.binary_cross_entropy_with_logits(s_i - s_j, p_target)\n\n\ndef pointwise_mse(s: torch.Tensor, y: torch.Tensor) -> torch.Tensor:\n    """Objektif pointwise untuk B4."""\n    return F.mse_loss(s, y)\n',
    'src/hr/pairs.py': '"""Pembentukan pasangan pelatihan (persamaan 3.5).\n\nPasangan (i, j) hanya dibentuk di dalam produk yang sama dengan y_i - y_j > delta. Setiap produk\ndibatasi paling banyak M pasangan yang diambil acak, dan pasangan diambil ulang setiap epoch.\nUntuk T2 (age-matched pairs), selisih ln(1 + usia) kedua ulasan harus di bawah ``age_window``.\n"""\nfrom __future__ import annotations\n\nimport numpy as np\n\n\nclass PairSampler:\n    def __init__(self, pid: np.ndarray, y: np.ndarray, max_pairs: int, delta: float = 0.0,\n                 log_age: np.ndarray | None = None, age_window: float | None = None, seed: int = 0,\n                 oversample: int = 4):\n        self.y = np.asarray(y, dtype=np.float64)\n        self.max_pairs = int(max_pairs)\n        self.delta = float(delta)\n        self.log_age = None if log_age is None else np.asarray(log_age, dtype=np.float64)\n        self.age_window = age_window\n        self.rng = np.random.default_rng(seed)\n        self.oversample = oversample\n        codes = np.unique(np.asarray(pid), return_inverse=True)[1]\n        self.order = np.argsort(codes, kind="stable")      # posisi baris dikelompokkan per produk\n        sizes = np.bincount(codes)\n        starts = np.r_[0, np.cumsum(sizes)[:-1]]\n        ok = sizes >= 2\n        self.sizes, self.starts = sizes[ok], starts[ok]\n        self.k = np.minimum(self.max_pairs, self.sizes * (self.sizes - 1) // 2)\n\n    def sample(self) -> tuple[np.ndarray, np.ndarray]:\n        """Kembalikan (hi, lo): posisi baris dengan y[hi] > y[lo] + delta."""\n        d = np.maximum(self.k * self.oversample, 4)\n        g = np.repeat(np.arange(len(self.sizes)), d)\n        s = self.sizes[g]\n        a = (self.rng.random(len(g)) * s).astype(np.int64)\n        b = (self.rng.random(len(g)) * (s - 1)).astype(np.int64)\n        b += b >= a                                          # pastikan a != b\n        ia = self.order[self.starts[g] + a]\n        ib = self.order[self.starts[g] + b]\n        diff = self.y[ia] - self.y[ib]\n        keep = np.abs(diff) > self.delta\n        if self.log_age is not None and self.age_window is not None:\n            keep &= np.abs(self.log_age[ia] - self.log_age[ib]) < self.age_window\n        ia, ib, diff, g = ia[keep], ib[keep], diff[keep], g[keep]\n        hi = np.where(diff > 0, ia, ib)\n        lo = np.where(diff > 0, ib, ia)\n        # hapus duplikat, lalu ambil paling banyak k pasangan per produk\n        n = len(self.y) + 1\n        key = hi.astype(np.int64) * n + lo\n        _, first = np.unique(key, return_index=True)\n        hi, lo, g = hi[first], lo[first], g[first]\n        perm = self.rng.permutation(len(g))\n        hi, lo, g = hi[perm], lo[perm], g[perm]\n        srt = np.argsort(g, kind="stable")\n        hi, lo, g = hi[srt], lo[srt], g[srt]\n        grp_start = np.r_[0, np.flatnonzero(np.diff(g)) + 1]\n        rank = np.arange(len(g)) - np.repeat(grp_start, np.diff(np.r_[grp_start, len(g)]))\n        take = rank < self.k[g]\n        hi, lo = hi[take], lo[take]\n        perm = self.rng.permutation(len(hi))\n        return hi[perm], lo[perm]\n\n    def stats(self, pid: np.ndarray) -> dict:\n        """Ringkasan satu kali pengambilan pasangan, untuk dicatat di Bab IV."""\n        hi, lo = self.sample()\n        pid = np.asarray(pid)\n        return {"pairs": int(len(hi)),\n                "products_with_pairs": int(len(np.unique(pid[hi]))),\n                "products_eligible": int(len(self.sizes)),\n                "mean_target_gap": float(np.mean(self.y[hi] - self.y[lo])) if len(hi) else float("nan")}\n',
    'src/hr/plots.py': '"""Grafik untuk Bab IV. Semua gambar disimpan 300 dpi sesuai syarat JuTISI."""\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\n\nplt.rcParams.update({"font.family": "serif", "font.size": 10, "figure.dpi": 110,\n                     "axes.spines.top": False, "axes.spines.right": False})\n\n\ndef save(fig, path: Path) -> Path:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    fig.savefig(path, dpi=300, bbox_inches="tight")\n    return path\n\n\ndef vote_distribution(votes: np.ndarray, ax=None):\n    ax = ax or plt.subplots(figsize=(5, 3.2))[1]\n    v = np.asarray(votes)\n    bins = np.r_[0, np.unique(np.round(np.logspace(0, np.log10(max(v.max(), 2)), 30))), v.max() + 1]\n    ax.hist(v, bins=bins, color="#4C72B0")\n    ax.set_xscale("symlog")\n    ax.set_yscale("log")\n    ax.set_xlabel("helpful_vote")\n    ax.set_ylabel("jumlah ulasan")\n    ax.set_title(f"Sebaran helpful_vote (nol = {np.mean(v == 0):.1%})")\n    return ax\n\n\ndef simple_hist(x, xlabel: str, title: str, log_y: bool = False, bins: int = 50, ax=None):\n    ax = ax or plt.subplots(figsize=(5, 3.2))[1]\n    ax.hist(np.asarray(x), bins=bins, color="#4C72B0")\n    if log_y:\n        ax.set_yscale("log")\n    ax.set_xlabel(xlabel)\n    ax.set_ylabel("frekuensi")\n    ax.set_title(title)\n    return ax\n\n\ndef age_curve(ages, curve, lo=None, hi=None, ref_days: float = 30.0, ax=None):\n    ax = ax or plt.subplots(figsize=(5, 3.2))[1]\n    ax.plot(ages, curve, color="#C44E52")\n    if lo is not None:\n        ax.fill_between(ages, lo, hi, color="#C44E52", alpha=0.2, label="IK 95%")\n        ax.legend()\n    ax.axhline(1.0, color="grey", lw=0.8, ls="--")\n    ax.set_xscale("log")\n    ax.set_yscale("log")        # rasio membentang beberapa orde besaran; skala linear menyembunyikan bentuknya\n    ax.set_xlabel("usia ulasan (hari)")\n    ax.set_ylabel(f"rasio rata-rata suara vs usia {ref_days:.0f} hari")\n    ax.set_title("Hubungan usia ulasan dan helpful votes")\n    return ax\n\n\ndef _ci_cols(tab: pd.DataFrame) -> pd.DataFrame:\n    return tab.rename(columns={"ndcg@10": "mean", "ndcg@10_lo": "lo", "ndcg@10_hi": "hi"})\n\n\ndef baseline_figure(main_tab: pd.DataFrame, r0_tab: pd.DataFrame | None = None, highlight: str = "U"):\n    """Grafik ``uji_baseline.png``: NDCG@10 uji dengan IK 95%, ditambah urutan acak R0 (abu-abu) jika ada."""\n    t = _ci_cols(main_tab)\n    if r0_tab is not None:\n        t = pd.concat([_ci_cols(r0_tab), t], ignore_index=True)\n    ax = bar_ci(t, "code", "NDCG@10 uji (IK 95%)", highlight=highlight)\n    if r0_tab is not None:\n        ax.patches[0].set_color("#9A9A9A")\n    ax.set_title("Baseline dan model utama")\n    return ax\n\n\ndef age_figure(curve_tab: pd.DataFrame, group_tab: pd.DataFrame):\n    """Grafik ``usia.png``: kurva usia dengan IK, dan NDCG@10 per kelompok usia untuk T1, T2, T3\n    (protokol relevansi mentah)."""\n    fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))\n    c = curve_tab\n    age_curve(c["age_days"], c["ratio"], c["lo"], c["hi"], ax=axes[0])\n    g = group_tab[group_tab["protokol_relevansi"] == "T1"].rename(columns={"ndcg@10": "mean"})\n    grouped_bars(g, "group", "target", ax=axes[1])\n    axes[1].set_ylabel("NDCG@10 uji (relevansi mentah)")\n    axes[1].set_title("Strategi target per kelompok usia")\n    fig.tight_layout()\n    return fig\n\n\ndef loss_curves(hist: pd.DataFrame, title: str = "", axes=None):\n    if axes is None:\n        fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))\n    from matplotlib.ticker import MaxNLocator\n    axes[0].plot(hist["epoch"], hist["train_loss"], marker="o", label="latih (loss objektif model)")\n    axes[0].plot(hist["epoch"], hist["val_ranknet_loss"], marker="s", label="validasi (loss RankNet)")\n    axes[0].set_xlabel("epoch")\n    axes[0].set_ylabel("loss")\n    axes[0].legend(fontsize=8)\n    axes[1].plot(hist["epoch"], hist["val_ndcg@10"], marker="o", color="#55A868")\n    best = hist["val_ndcg@10"].idxmax()\n    axes[1].axvline(hist.loc[best, "epoch"], color="grey", ls="--", lw=0.8)\n    axes[1].set_xlabel("epoch")\n    axes[1].set_ylabel("NDCG@10 validasi")\n    for ax in axes:\n        ax.xaxis.set_major_locator(MaxNLocator(integer=True))\n    if title:\n        axes[0].figure.suptitle(title)\n    return axes\n\n\ndef bar_ci(tab: pd.DataFrame, label_col: str, metric_label: str = "NDCG@10", ax=None, highlight=None):\n    """``tab`` berisi kolom label, mean, lo, hi."""\n    ax = ax or plt.subplots(figsize=(6, 3.4))[1]\n    x = np.arange(len(tab))\n    colors = ["#C44E52" if highlight is not None and l == highlight else "#4C72B0" for l in tab[label_col]]\n    ax.bar(x, tab["mean"], color=colors)\n    ax.errorbar(x, tab["mean"], yerr=[tab["mean"] - tab["lo"], tab["hi"] - tab["mean"]], fmt="none",\n                ecolor="black", capsize=3, lw=1)\n    ax.set_xticks(x, tab[label_col])\n    ax.set_ylabel(metric_label)\n    lo = float(np.nanmin(tab["lo"]))\n    ax.set_ylim(max(0.0, lo - 0.05), None)\n    return ax\n\n\ndef scale_curve(tab: pd.DataFrame, ax=None):\n    """``tab`` berisi kolom reviews, mean, lo, hi."""\n    ax = ax or plt.subplots(figsize=(5, 3.2))[1]\n    ax.plot(tab["reviews"], tab["mean"], marker="o")\n    ax.fill_between(tab["reviews"], tab["lo"], tab["hi"], alpha=0.2)\n    ax.set_xscale("log")\n    ax.set_xlabel("jumlah ulasan latih")\n    ax.set_ylabel("NDCG@10 uji")\n    ax.set_title("Pengaruh ukuran data latih")\n    return ax\n\n\ndef grouped_bars(tab: pd.DataFrame, group_col: str, series_col: str, value_col: str = "mean", ax=None):\n    ax = ax or plt.subplots(figsize=(6, 3.4))[1]\n    groups = list(tab[group_col].unique())\n    series = list(tab[series_col].unique())\n    w = 0.8 / len(series)\n    for k, s in enumerate(series):\n        sub = tab[tab[series_col] == s].set_index(group_col).reindex(groups)\n        ax.bar(np.arange(len(groups)) + k * w - 0.4 + w / 2, sub[value_col], width=w, label=str(s))\n    ax.set_xticks(np.arange(len(groups)), groups)\n    ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), frameon=False)\n    return ax\n\n\ndef size_figure(size_tab: pd.DataFrame, model: str = "U"):\n    """Grafik ``uji_ukuran_produk.png``: NDCG@10 model per kuartil ukuran produk, di samping urutan acak."""\n    t = pd.concat([pd.DataFrame({"kelompok": size_tab["kelompok"], "seri": model, "mean": size_tab["ndcg@10"]}),\n                   pd.DataFrame({"kelompok": size_tab["kelompok"], "seri": "R0 (acak)",\n                                 "mean": size_tab["ndcg@10_acak"]})], ignore_index=True)\n    ax = grouped_bars(t, "kelompok", "seri", ax=plt.subplots(figsize=(6, 3.4))[1])\n    ax.set_ylabel("NDCG@10 uji")\n    ax.set_title(f"Model {model} dan urutan acak menurut ukuran produk")\n    return ax\n',
    'src/hr/report.py': '"""Evaluasi data uji dan penyusunan tabel Bab IV (Subbab 3.9 dan 3.10)."""\nfrom __future__ import annotations\n\nimport re\n\nimport numpy as np\nimport pandas as pd\n\nfrom .experiment import Bundle, load_pred, protocol_cols\nfrom .metrics import macro, per_product_metrics\nfrom .stats import average_over_seeds, holm, paired_diff, summarize\n\n\ndef eval_run(b: Bundle, run: str, split: str = "test", protocol: str = "T1",\n             rows: pd.DataFrame | None = None) -> pd.DataFrame:\n    """Metrik per produk untuk satu run. ``rows`` dapat dipakai untuk subset (misalnya kelompok usia)."""\n    rows = b.rows(split) if rows is None else rows\n    pred = load_pred(b.cfg, run, split)\n    d = rows.merge(pred, on="rid", how="inner")\n    if len(d) != len(rows):\n        raise ValueError(f"Run {run}: prediksi {split} hanya mencakup {len(d)} dari {len(rows)} ulasan. "\n                         "Run dibuat dari pembagian data lain, jalankan ulang run tersebut.")\n    if not np.isfinite(d["score"].to_numpy(dtype=np.float64)).all():\n        raise ValueError(f"Run {run}: ada skor NaN/inf pada prediksi {split}.")\n    g, t = protocol_cols(protocol)\n    return per_product_metrics(d["parent_asin"].to_numpy(), d["score"].to_numpy(), d[g].to_numpy(),\n                               d[t].to_numpy(), d["rid"].to_numpy(), map_rel=b.cfg.map_relevant_grade)\n\n\ndef eval_config(b: Bundle, runs: list[str], split: str = "test", protocol: str = "T1",\n                rows: pd.DataFrame | None = None) -> pd.DataFrame:\n    """Rata-rata metrik per produk dari semua seed satu konfigurasi."""\n    return average_over_seeds([eval_run(b, r, split, protocol, rows) for r in runs])\n\n\ndef expected_runs(cfg) -> dict[tuple, set[str]]:\n    """Semua run final yang seharusnya sudah dibuat notebook 05-06 sesuai ``run_priority``,\n    dikelompokkan per (code, scale, target)."""\n    from .experiment import run_id\n    MS, TG = cfg.main_scale, cfg.main_target\n    todo = cfg.experiments_to_run()\n    seeded = lambda c, sc, t: {run_id(c, sc, t, s) for s in cfg.seeds}\n    exp = {("B0", MS, TG): {run_id("B0", MS, TG, 0)}}\n    if "B1" in todo:\n        exp[("B1", MS, TG)] = {run_id("B1", MS, TG, cfg.seed)}\n    for c in ["U", "B2", "B3", "B4", "B5", "A1", "A2", "A3"]:\n        if c in todo:\n            exp[(c, MS, TG)] = seeded(c, MS, TG)\n    for sc in cfg.scales_to_run():\n        exp[("U", sc, TG)] = seeded("U", sc, TG)\n    for t in cfg.targets_to_run():\n        if t in ("T2", "T3"):                     # T1 = target utama, sudah tercakup di atas\n            exp[("U", MS, t)] = seeded("U", MS, t)\n    return exp\n\n\ndef check_complete(cfg, groups: dict) -> None:\n    """Gagal keras jika run yang dijadwalkan belum ada, atau ada run seed lain (dari konfigurasi lama)\n    yang akan ikut dirata-rata. Dengan begitu tabel Bab IV tidak diam-diam kehilangan atau\n    kelebihan baris."""\n    problems = []\n    for k, want in expected_runs(cfg).items():\n        have = set(groups.get(k, []))\n        if have != want:\n            problems.append(f"{k}: kurang {sorted(want - have)}, kelebihan {sorted(have - want)}")\n    if problems:\n        raise RuntimeError("Run tidak sesuai jadwal. Jalankan notebook 05-06, atau hapus folder run yang "\n                           "kelebihan:\\n  " + "\\n  ".join(problems))\n    print("Semua", len(expected_runs(cfg)), "konfigurasi yang dijadwalkan lengkap.")\n\n\ndef group_runs(results: pd.DataFrame) -> dict[tuple, list[str]]:\n    """Kelompokkan run_id berdasarkan (code, scale, target) dengan mengabaikan seed dan run penalaan."""\n    res = results[~results["run_id"].str.contains("_tune")]\n    return {k: list(g["run_id"]) for k, g in res.groupby(["code", "scale", "target"])}\n\n\ndef comparison_table(b: Bundle, groups: dict, keys: list[tuple], protocol: str = "T1", n_boot: int = 1000,\n                     split: str = "test") -> tuple[pd.DataFrame, dict]:\n    """Tabel metrik dengan IK 95% untuk beberapa konfigurasi. Mengembalikan juga metrik per produk."""\n    rows, pps = [], {}\n    missing = [k for k in keys if k not in groups]\n    if missing:\n        print("Dilewati (tidak ada run):", missing)\n    for key in keys:\n        if key not in groups:\n            continue\n        pp = eval_config(b, groups[key], split, protocol)\n        pps[key] = pp\n        s = summarize(pp, n_boot)\n        r = {"code": key[0], "scale": key[1], "target": key[2], "seeds": len(groups[key])}\n        for _, m in s.iterrows():\n            r[m["metric"]] = m["mean"]\n            r[f"{m[\'metric\']}_lo"] = m["lo"]\n            r[f"{m[\'metric\']}_hi"] = m["hi"]\n            r[f"n_{m[\'metric\']}"] = m["n"]\n        rows.append(r)\n    return pd.DataFrame(rows), pps\n\n\ndef paired_table(pps: dict, ref: tuple, others: list[tuple], metric: str = "ndcg@10", n_boot: int = 1000) -> pd.DataFrame:\n    rows = []\n    for o in others:\n        if o not in pps or ref not in pps:\n            continue\n        d = paired_diff(pps[ref], pps[o], metric, n_boot)\n        rows.append({"ref": ref[0], "vs": o[0], **d})\n    tab = pd.DataFrame(rows)\n    if len(tab):\n        tab["p_holm"] = holm(tab["p"].to_numpy())\n        tab["lebih_baik"] = (tab["lo"] > 0) & (tab["p_holm"] < 0.05)\n    return tab\n\n\ndef _seed_of(run: str) -> int:\n    m = re.search(r"_s(\\d+)$", run)\n    if not m:\n        raise ValueError(f"run_id {run} tidak berakhiran _s<seed>")\n    return int(m.group(1))\n\n\ndef seed_pairs(ref_runs: list[str], other_runs: list[str]) -> list[tuple[str, str]]:\n    """Pasangan run (ref, pembanding) per seed. Pembanding dengan satu run saja (B0 heuristik, B1\n    deterministik) dipasangkan dengan setiap seed ref. Selain itu seed harus sama persis."""\n    ref = sorted(ref_runs, key=_seed_of)\n    if len(other_runs) == 1:\n        return [(r, other_runs[0]) for r in ref]\n    by_seed = {_seed_of(r): r for r in other_runs}\n    if sorted(by_seed) != sorted(_seed_of(r) for r in ref):\n        raise ValueError(f"Seed tidak sama: {sorted(map(_seed_of, ref))} vs {sorted(by_seed)}")\n    return [(r, by_seed[_seed_of(r)]) for r in ref]\n\n\ndef add_seed_consistency(tab: pd.DataFrame, b: Bundle, groups: dict, ref: tuple, others: list[tuple],\n                         metric: str = "ndcg@10", protocol: str = "T1", split: str = "test") -> pd.DataFrame:\n    """Tambahkan konsistensi per seed ke tabel ``paired_table`` (pelengkap aturan "lebih baik", bukan\n    syaratnya). Untuk setiap pasangan seed dihitung selisih rata-rata ``metric`` per produk pada produk\n    yang valid untuk keduanya, sama seperti ``paired_diff`` tetapi tanpa merata-ratakan seed.\n\n    Kolom baru: ``selisih_per_seed`` (teks), ``seed_searah`` (berapa pasangan seed yang arahnya sama\n    dengan ``diff`` gabungan), ``n_pasangan_seed``, dan ``searah_semua_seed``."""\n    cache: dict[str, pd.DataFrame] = {}\n\n    def pp(run):\n        if run not in cache:\n            cache[run] = eval_run(b, run, split, protocol)\n        return cache[run]\n\n    extra = []\n    for o in others:\n        if o not in groups or ref not in groups:\n            continue\n        diffs = []\n        for ra, rb in seed_pairs(groups[ref], groups[o]):\n            m = pp(ra)[["parent_asin", metric]].merge(pp(rb)[["parent_asin", metric]], on="parent_asin",\n                                                      suffixes=("_a", "_b")).dropna()\n            diffs.append((_seed_of(ra), float((m[f"{metric}_a"] - m[f"{metric}_b"]).mean())))\n        extra.append({"vs": o[0], "_diffs": diffs})\n    ex = pd.DataFrame(extra)\n    out = tab.merge(ex, on="vs", how="left", validate="one_to_one")\n    if out["_diffs"].isna().any():\n        raise ValueError(f"Konsistensi seed tidak bisa dihitung untuk {out.loc[out[\'_diffs\'].isna(), \'vs\'].tolist()}")\n    sign = np.sign(out["diff"].to_numpy())\n    out["selisih_per_seed"] = [" | ".join(f"s{s} {d:+.4f}" for s, d in ds) for ds in out["_diffs"]]\n    out["seed_searah"] = [int(sum(np.sign(d) == sg for _, d in ds)) for ds, sg in zip(out["_diffs"], sign)]\n    out["n_pasangan_seed"] = [len(ds) for ds in out["_diffs"]]\n    out["searah_semua_seed"] = out["seed_searah"] == out["n_pasangan_seed"]\n    return out.drop(columns="_diffs")\n\n\ndef age_groups(b: Bundle, edges: list[float]) -> pd.Series:\n    """Kelompok usia ulasan (baru, menengah, lama) dengan batas dari data latih."""\n    return pd.cut(b.df["age_days"], [-np.inf, edges[0], edges[1], np.inf], labels=["baru", "menengah", "lama"])\n\n\ndef by_subset(b: Bundle, runs: list[str], subsets: dict[str, pd.DataFrame], protocol: str = "T1") -> pd.DataFrame:\n    rows = []\n    for name, sub in subsets.items():\n        pp = eval_config(b, runs, "test", protocol, sub)\n        rows.append({"group": name, **macro(pp)})\n    return pd.DataFrame(rows)\n\n\ndef pointwise_errors(b: Bundle, runs: list[str]) -> dict:\n    """MAE dan RMSE terhadap ln(1 + v) khusus pembanding pointwise B4."""\n    te = b.rows("test")\n    errs = []\n    for r in runs:\n        d = te.merge(load_pred(b.cfg, r, "test"), on="rid")\n        e = d["score"].to_numpy() - d["y_t1"].to_numpy()\n        errs.append((np.abs(e).mean(), np.sqrt((e ** 2).mean())))\n    e = np.array(errs)\n    return {"mae": float(e[:, 0].mean()), "rmse": float(e[:, 1].mean()), "seeds": len(runs)}\n\n\nR0_PERMUTATIONS = 20   # jumlah permutasi acak untuk baseline R0 (notebook 07 dan 00b)\n\n\ndef random_per_product(b: Bundle, protocol: str = "T1", split: str = "test", n_perm: int = 20,\n                       seed: int = 0, rows: pd.DataFrame | None = None) -> pd.DataFrame:\n    """Metrik per produk untuk urutan acak (R0), dirata-rata atas ``n_perm`` permutasi.\n\n    Ini batas bawah yang harus dilampaui model. Pada produk dengan sedikit ulasan, NDCG@10 urutan\n    acak pun sudah tinggi, jadi angka model hanya bermakna bila dibandingkan dengan R0."""\n    rows = b.rows(split) if rows is None else rows\n    g, t = protocol_cols(protocol)\n    rng = np.random.default_rng(seed)\n    pps = [per_product_metrics(rows["parent_asin"].to_numpy(), rng.random(len(rows)), rows[g].to_numpy(),\n                               rows[t].to_numpy(), rows["rid"].to_numpy(), map_rel=b.cfg.map_relevant_grade)\n           for _ in range(n_perm)]\n    return average_over_seeds(pps)\n\n\ndef random_table(pp_r0: pd.DataFrame, n_perm: int, n_boot: int = 1000) -> pd.DataFrame:\n    """Satu baris R0 dengan kolom yang sama seperti ``comparison_table`` (``tables/uji_acak.csv``)."""\n    r = {"code": "R0 (acak)", "scale": None, "target": None, "seeds": n_perm}\n    for _, m in summarize(pp_r0, n_boot).iterrows():\n        r[m["metric"]], r[f"{m[\'metric\']}_lo"], r[f"{m[\'metric\']}_hi"], r[f"n_{m[\'metric\']}"] = \\\n            m["mean"], m["lo"], m["hi"], m["n"]\n    return pd.DataFrame([r])\n\n\ndef size_groups(rows: pd.DataFrame) -> pd.Series:\n    """Kuartil ukuran produk (jumlah ulasan di ``rows``), indeks parent_asin."""\n    size = rows.groupby("parent_asin").size()\n    return pd.qcut(size.rank(method="first"), 4, labels=["Q1 kecil", "Q2", "Q3", "Q4 besar"]).rename("kelompok")\n\n\ndef size_table(pp_model: pd.DataFrame, pp_r0: pd.DataFrame, rows: pd.DataFrame) -> pd.DataFrame:\n    """Metrik model per kuartil ukuran produk, berdampingan dengan urutan acak R0\n    (``tables/uji_per_ukuran_produk.csv``)."""\n    grp = size_groups(rows).reset_index()\n    cols = ["ndcg@10", "spearman", "ap"]\n    m = pp_model.merge(grp, on="parent_asin").groupby("kelompok", observed=True)[cols].mean()\n    r = pp_r0.merge(grp, on="parent_asin").groupby("kelompok", observed=True)[["ndcg@10", "ap"]].mean()\n    out = m.join(r.add_suffix("_acak"))\n    out["n_produk"] = grp.groupby("kelompok", observed=True).size()\n    out["median_ulasan"] = rows.groupby("parent_asin").size().groupby(size_groups(rows), observed=True).median()\n    return out.round(4)\n',
    'src/hr/split.py': '"""Pembagian data product-disjoint dan subset skala E1 sampai E5 (Subbab 3.4)."""\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom .config import STRUCT_FEATURES, Config\n\n\ndef split_products(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:\n    """Acak daftar parent_asin dengan seed tetap, lalu alokasikan 80/10/10 per produk."""\n    rng = np.random.default_rng(cfg.seed)\n    prods = np.sort(df["parent_asin"].unique())\n    perm = rng.permutation(len(prods))\n    n_tr = int(round(cfg.split_frac[0] * len(prods)))\n    n_va = int(round(cfg.split_frac[1] * len(prods)))\n    split = np.empty(len(prods), dtype=object)\n    split[perm[:n_tr]] = "train"\n    split[perm[n_tr:n_tr + n_va]] = "val"\n    split[perm[n_tr + n_va:]] = "test"\n    m = dict(zip(prods, split))\n    out = df.copy()\n    out["split"] = out["parent_asin"].map(m)\n    return out\n\n\ndef leakage_checks(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:\n    """Pemeriksaan otomatis setelah alokasi. Baris validasi/uji yang bocor dibuang, baris latih\n    tidak diubah."""\n    rep = {}\n    by = {s: set(df.loc[df["split"] == s, "parent_asin"]) for s in ("train", "val", "test")}\n    rep["parent_overlap"] = {\n        "train_val": len(by["train"] & by["val"]),\n        "train_test": len(by["train"] & by["test"]),\n        "val_test": len(by["val"] & by["test"]),\n    }\n    assert sum(rep["parent_overlap"].values()) == 0, "parent_asin muncul di dua partisi"\n\n    asin_splits = df.groupby("asin")["split"].nunique()\n    shared_asin = set(asin_splits[asin_splits > 1].index)\n    rep["asin_in_multiple_splits"] = len(shared_asin)\n\n    train_hash = set(df.loc[df["split"] == "train", "text_hash"])\n    leak = (df["split"] != "train") & (df["asin"].isin(shared_asin) | df["text_hash"].isin(train_hash))\n    rep["eval_rows_dropped_for_leakage"] = int(leak.sum())\n    df = df[~leak]\n    # buang lagi produk evaluasi yang tinggal < 2 ulasan\n    sizes = df.groupby("parent_asin")["rid"].transform("size")\n    small = (df["split"] != "train") & (sizes < 2)\n    rep["eval_rows_dropped_small_after_leak"] = int(small.sum())\n    return df[~small].copy(), rep\n\n\ndef cap_eval(df: pd.DataFrame, cfg: Config) -> tuple[pd.DataFrame, dict]:\n    """Batasi ukuran validasi/uji dengan mengambil produk utuh secara acak."""\n    rep = {}\n    rng = np.random.default_rng(cfg.seed + 1)\n    keep = np.ones(len(df), dtype=bool)\n    for s in ("val", "test"):\n        sub = df[df["split"] == s]\n        if len(sub) <= cfg.max_eval_reviews:\n            rep[s] = {"products": int(sub["parent_asin"].nunique()), "reviews": len(sub), "capped": False}\n            continue\n        sizes = sub.groupby("parent_asin").size()\n        order = rng.permutation(sizes.index.to_numpy())\n        cum = sizes.loc[order].cumsum()\n        chosen = set(cum[cum <= cfg.max_eval_reviews].index)\n        drop = (df["split"] == s) & ~df["parent_asin"].isin(chosen)\n        keep &= ~drop.to_numpy()\n        rep[s] = {"products": len(chosen), "reviews": int(sizes.loc[list(chosen)].sum()), "capped": True}\n    return df[keep].copy(), rep\n\n\ndef make_scales(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:\n    """Subset bertingkat (nested). Produk latih diacak sekali, setiap produk dibatasi paling banyak\n    ``max_train_reviews_per_product`` ulasan, lalu skala Ek diambil sebagai awalan urutan yang sama.\n    Karena itu E1 selalu bagian dari E2, E2 bagian dari E3, dan seterusnya."""\n    rng = np.random.default_rng(cfg.seed + 2)\n    tr = df[df["split"] == "train"][["rid", "parent_asin"]].copy()\n    tr["u"] = rng.random(len(tr))\n    tr = tr.sort_values(["parent_asin", "u"])\n    tr["k"] = tr.groupby("parent_asin").cumcount()\n    tr = tr[tr["k"] < cfg.max_train_reviews_per_product]\n    prods = tr["parent_asin"].unique()\n    prod_rank = dict(zip(prods[rng.permutation(len(prods))], range(len(prods))))\n    tr["prank"] = tr["parent_asin"].map(prod_rank)\n    tr = tr.sort_values(["prank", "k"]).reset_index(drop=True)\n    tr["order"] = np.arange(len(tr))\n    out = df.copy()\n    pos = pd.Series(tr["order"].to_numpy(), index=tr["rid"].to_numpy())\n    out["train_order"] = out["rid"].map(pos)          # NaN untuk validasi/uji atau yang terpotong batas\n    short = {name: n for name, n in cfg.scales.items() if n > len(tr)}\n    if short:\n        # Tanpa ini, skala besar diam-diam sama dengan skala di bawahnya dan kurva skala (RM2) menjadi datar.\n        raise ValueError(f"Data latih hanya {len(tr)} ulasan (setelah batas {cfg.max_train_reviews_per_product} "\n                         f"per produk), tidak cukup untuk {short}. Pilih kategori lebih besar atau ubah cfg.scales.")\n    for name, n in cfg.scales.items():\n        col = out["train_order"] < n\n        # buang produk yang hanya terambil satu ulasan di ujung awalan\n        cnt = out[col].groupby("parent_asin")["rid"].transform("size")\n        single = cnt[cnt < 2].index\n        col.loc[single] = False\n        out[f"in_{name}"] = col.fillna(False).astype(bool)\n    return out\n\n\ndef scale_report(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:\n    rows = []\n    for name in cfg.scales:\n        sub = df[df[f"in_{name}"]]\n        rows.append({"scale": name, "reviews": len(sub), "products": sub["parent_asin"].nunique(),\n                     "zero_vote_share": float((sub["helpful_vote"] == 0).mean())})\n    for s in ("val", "test"):\n        sub = df[df["split"] == s]\n        rows.append({"scale": s, "reviews": len(sub), "products": sub["parent_asin"].nunique(),\n                     "zero_vote_share": float((sub["helpful_vote"] == 0).mean())})\n    return pd.DataFrame(rows)\n\n\ndef fit_standardizer(df: pd.DataFrame) -> dict:\n    """Persamaan (3.2). Rata-rata dan simpangan baku hanya dari data latih."""\n    tr = df[df["split"] == "train"]\n    return {c: (float(tr[c].mean()), float(tr[c].std() or 1.0)) for c in STRUCT_FEATURES}\n\n\ndef apply_standardizer(df: pd.DataFrame, stats: dict) -> pd.DataFrame:\n    out = df.copy()\n    for c, (m, s) in stats.items():\n        out[f"z_{c}"] = ((out[c].astype(np.float64) - m) / s).astype(np.float32)\n    return out\n',
    'src/hr/stages.py': '"""Menjalankan seluruh tahap dalam satu notebook (``00_pipeline_lengkap.ipynb``).\n\nSetiap sel kode tahap diawali ``%%tahap N``. Magic ini:\n- melewati sel jika tahap N tidak dipilih (``mulai_dari_tahap`` dan kotak centang tahap opsional);\n- saat berpindah ke tahap baru, menghapus variabel tahap sebelumnya dan cache keluaran agar RAM\n  tidak menumpuk. Notebook 01-10 aslinya berjalan di kernel terpisah, dan setiap tahap memuat\n  ulang masukannya dari penyimpanan, jadi tidak ada tahap yang bergantung pada variabel tahap lain;\n- jika sel gagal, menaikkan error sehingga "Run all" berhenti di sel itu, tidak lanjut diam-diam.\n"""\nfrom __future__ import annotations\n\nimport gc\nimport sys\n\n\ndef setup(start: int, optional: dict[int, bool] | None = None) -> None:\n    """Daftarkan magic ``%%tahap``. Panggil di baris TERAKHIR sel pengaturan: semua nama yang sudah\n    ada saat itu (cfg, np, pd, ...) dipertahankan ketika berpindah tahap."""\n    from IPython import get_ipython\n    ip = get_ipython()\n    if ip is None:\n        raise RuntimeError("hr.stages hanya bisa dipakai di dalam notebook.")\n    optional = optional or {}\n    keep = set(ip.user_ns) | {"_", "__", "___"}\n    state = {"stage": None}\n\n    def selected(n: int) -> bool:\n        return n >= start and optional.get(n, True)\n\n    def clean() -> None:\n        ns = ip.user_ns\n        for name in [k for k in ns if k not in keep]:\n            del ns[name]\n        ip.run_line_magic("reset", "-f out")       # cache Out[] menahan DataFrame besar\n        gc.collect()\n        if "torch" in sys.modules and sys.modules["torch"].cuda.is_available():\n            sys.modules["torch"].cuda.empty_cache()\n\n    def tahap(line: str, cell: str) -> None:\n        n = int(line.split()[0])\n        if not selected(n):\n            print(f"[tahap {n:02d} dilewati]")\n            return\n        if state["stage"] != n:\n            clean()\n            state["stage"] = n\n        res = ip.run_cell(cell)\n        if not res.success:\n            raise RuntimeError(f"Tahap {n:02d} gagal (lihat error di atas). Perbaiki penyebabnya lalu jalankan "\n                               "ulang dari sel ini, atau atur mulai_dari_tahap dan jalankan semua.") from None\n\n    ip.register_magic_function(tahap, magic_kind="cell", magic_name="tahap")\n    chosen = [n for n in range(1, 11) if selected(n)]\n    print("tahap yang akan dijalankan:", chosen)\n',
    'src/hr/stats.py': '"""Ketidakpastian dan perbandingan model (Subbab 3.9).\n\nBootstrap dilakukan pada tingkat produk (parent_asin) sehingga ulasan dalam satu produk tetap\nterkumpul. Perbandingan dua model memakai selisih metrik per produk pada produk yang valid untuk\nkedua model.\n"""\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\n\ndef bootstrap_mean(values: np.ndarray, n_boot: int = 1000, seed: int = 0, alpha: float = 0.05) -> dict:\n    v = np.asarray(values, dtype=np.float64)\n    v = v[~np.isnan(v)]\n    if len(v) == 0:\n        return {"mean": np.nan, "lo": np.nan, "hi": np.nan, "n": 0}\n    rng = np.random.default_rng(seed)\n    idx = rng.integers(0, len(v), size=(n_boot, len(v)))\n    means = v[idx].mean(axis=1)\n    return {"mean": float(v.mean()), "lo": float(np.quantile(means, alpha / 2)),\n            "hi": float(np.quantile(means, 1 - alpha / 2)), "n": int(len(v))}\n\n\ndef summarize(per_prod: pd.DataFrame, n_boot: int = 1000, seed: int = 0,\n              metrics=("ndcg@10", "ndcg@5", "spearman", "ap")) -> pd.DataFrame:\n    rows = []\n    for m in metrics:\n        r = bootstrap_mean(per_prod[m].to_numpy(), n_boot, seed)\n        rows.append({"metric": "map" if m == "ap" else m, **r})\n    return pd.DataFrame(rows)\n\n\ndef paired_diff(pp_a: pd.DataFrame, pp_b: pd.DataFrame, metric: str = "ndcg@10", n_boot: int = 1000,\n                seed: int = 0) -> dict:\n    """Selisih rata-rata (A - B) dengan IK 95% bootstrap dan p-value dua sisi."""\n    m = pp_a[["parent_asin", metric]].merge(pp_b[["parent_asin", metric]], on="parent_asin",\n                                            suffixes=("_a", "_b")).dropna()\n    d = (m[f"{metric}_a"] - m[f"{metric}_b"]).to_numpy()\n    if len(d) == 0:\n        return {"diff": np.nan, "lo": np.nan, "hi": np.nan, "p": np.nan, "n": 0}\n    rng = np.random.default_rng(seed)\n    boots = d[rng.integers(0, len(d), size=(n_boot, len(d)))].mean(axis=1)\n    p = 2 * min((boots <= 0).mean(), (boots >= 0).mean())\n    return {"diff": float(d.mean()), "lo": float(np.quantile(boots, 0.025)),\n            "hi": float(np.quantile(boots, 0.975)), "p": float(min(p, 1.0)), "n": int(len(d)),\n            "win": int((d > 0).sum()), "tie": int((d == 0).sum()), "loss": int((d < 0).sum())}\n\n\ndef holm(pvals) -> np.ndarray:\n    """Koreksi Holm untuk perbandingan berganda (kritik reviewer nomor 8)."""\n    p = np.asarray(pvals, dtype=np.float64)\n    order = np.argsort(p)\n    adj = np.empty_like(p)\n    running = 0.0\n    for rank, i in enumerate(order):\n        running = max(running, (len(p) - rank) * p[i])\n        adj[i] = min(running, 1.0)\n    return adj\n\n\ndef average_over_seeds(pps: list[pd.DataFrame]) -> pd.DataFrame:\n    """Rata-rata metrik per produk dari beberapa seed sebelum bootstrap."""\n    cat = pd.concat(pps, ignore_index=True)\n    return cat.groupby("parent_asin", as_index=False).mean(numeric_only=True)\n\n\ndef groupwise(per_prod: pd.DataFrame, group: pd.Series, metric: str = "ndcg@10") -> pd.DataFrame:\n    """Rata-rata metrik per kelompok (misalnya ukuran produk)."""\n    d = per_prod.merge(group.rename("group").reset_index(), on="parent_asin")\n    return d.groupby("group")[metric].agg(["mean", "count"]).reset_index()\n',
    'src/hr/synthetic.py': '"""Data sintetis berformat Amazon Reviews 2023, khusus untuk uji cepat (HR_SMOKE=1)."""\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nWORDS = ("good bad great battery screen fast slow cheap price quality broke works love hate return "\n         "size fits sound game controller disc graphics fun boring update").split()\n\n\ndef synthetic(n_products: int = 400, seed: int = 0) -> pd.DataFrame:\n    rng = np.random.default_rng(seed)\n    rows = []\n    t_end = pd.Timestamp("2023-09-01").value // 10 ** 6\n    for p in range(n_products):\n        n = int(rng.integers(2, 40))\n        pop = rng.normal(0, 1)\n        for i in range(n):\n            age = float(rng.exponential(700))\n            length = int(rng.integers(3, 120))\n            text = " ".join(rng.choice(WORDS, size=length))\n            mu = np.exp(-1.0 + pop + 0.35 * np.log1p(age) - 1.2 + 0.4 * np.log1p(length) / 3)\n            votes = int(rng.negative_binomial(2, 2 / (2 + mu)))\n            rows.append({"rating": float(rng.integers(1, 6)), "title": " ".join(rng.choice(WORDS, size=4)),\n                         "text": text + (" <br /> " if i % 7 == 0 else ""), "asin": f"A{p}-{i % 3}",\n                         "parent_asin": f"P{p}", "user_id": f"U{rng.integers(1e6)}",\n                         "timestamp": int(t_end - age * 86_400_000), "helpful_vote": votes,\n                         "verified_purchase": bool(rng.random() < 0.8)})\n    df = pd.DataFrame(rows)\n    return pd.concat([df, df.iloc[:5]], ignore_index=True)   # duplikat sengaja\n',
    'src/hr/targets.py': '"""Pembentukan target dan model hitung usia (Subbab 2.9 dan 3.5).\n\nModel hitung pada persamaan (2.17)::\n\n    ln mu_i = gamma_p(i) + f(ln(1 + a_i)) + beta^T c_i,   Var(v_i) = mu_i + kappa mu_i^2\n\ndiestimasi dengan maximum likelihood Negative Binomial (NB2). Efek produk gamma_p dimasukkan\nsebagai parameter tetap per produk (embedding), sehingga model tetap bisa dipakai untuk ratusan\nribu ulasan. Fungsi usia f memakai basis spline yang dipusatkan terhadap data latih model,\nsehingga f bernilai rata-rata nol dan konstanta seluruhnya diserap oleh efek produk.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.preprocessing import SplineTransformer\n\nCOVARIATES = ["rating", "verified_purchase", "log_review_length"]\n\n\n# ---------------------------------------------------------------- target mentah\ndef target_t1(votes) -> np.ndarray:\n    """Persamaan (2.15)."""\n    return np.log1p(np.asarray(votes, dtype=np.float64))\n\n\n# ---------------------------------------------------------------- basis usia\nclass AgeBasis:\n    """Basis fungsi usia f(ln(1 + a)). ``kind`` = "linear" atau jumlah knot spline."""\n\n    def __init__(self, kind):\n        self.kind = kind\n\n    def fit(self, log_age: np.ndarray) -> "AgeBasis":\n        x = np.asarray(log_age, dtype=np.float64).reshape(-1, 1)\n        if self.kind == "linear":\n            self.mu_, self.sd_ = float(x.mean()), float(x.std() or 1.0)\n            B = (x - self.mu_) / self.sd_\n        else:\n            self.spl_ = SplineTransformer(n_knots=int(self.kind), degree=3, extrapolation="linear",\n                                          include_bias=False)\n            B = self.spl_.fit_transform(x)\n        self.center_ = B.mean(axis=0)\n        return self\n\n    def transform(self, log_age: np.ndarray) -> np.ndarray:\n        x = np.asarray(log_age, dtype=np.float64).reshape(-1, 1)\n        B = (x - self.mu_) / self.sd_ if self.kind == "linear" else self.spl_.transform(x)\n        return (B - self.center_).astype(np.float32)\n\n\n# ---------------------------------------------------------------- NB2 dengan efek produk\ndef nb_loglik(y: torch.Tensor, mu: torch.Tensor, log_kappa: torch.Tensor) -> torch.Tensor:\n    r = torch.exp(-log_kappa)                     # r = 1 / kappa\n    return (torch.lgamma(y + r) - torch.lgamma(r) - torch.lgamma(y + 1)\n            + r * (torch.log(r) - torch.log(r + mu)) + y * (torch.log(mu) - torch.log(r + mu)))\n\n\ndef nb_deviance(y: np.ndarray, mu: np.ndarray, kappa: float) -> float:\n    y = np.asarray(y, dtype=np.float64)\n    mu = np.asarray(mu, dtype=np.float64)\n    r = 1.0 / kappa\n    with np.errstate(divide="ignore", invalid="ignore"):\n        t1 = np.where(y > 0, y * np.log(y / mu), 0.0)\n    t2 = (y + r) * np.log((y + r) / (mu + r))\n    return float(2.0 * np.sum(t1 - t2))\n\n\n@dataclass\nclass FitResult:\n    basis: AgeBasis | None\n    w: np.ndarray              # koefisien basis usia\n    beta: np.ndarray           # koefisien kovariat\n    cov_stats: dict            # standardisasi kovariat\n    kappa: float\n    loglik: float\n    deviance: float\n    gamma: dict                # efek produk (parent_asin -> nilai)\n\n    def f(self, log_age: np.ndarray) -> np.ndarray:\n        if self.basis is None:\n            return np.zeros(len(log_age), dtype=np.float64)\n        return self.basis.transform(log_age).astype(np.float64) @ self.w\n\n\ndef _cov_matrix(df: pd.DataFrame, stats: dict | None):\n    X = df[COVARIATES].to_numpy(dtype=np.float64)\n    if stats is None:\n        stats = {"mu": X.mean(0), "sd": X.std(0) + 1e-9}\n    return ((X - stats["mu"]) / stats["sd"]).astype(np.float32), stats\n\n\ndef fit_age_model(df: pd.DataFrame, basis_kind="linear", use_age: bool = True, steps: int = 400,\n                  lr: float = 0.05, l2_gamma: float = 1e-4, device: str | None = None,\n                  verbose: bool = False, basis: AgeBasis | None = None) -> FitResult:\n    """Estimasi NB2 dengan efek produk. ``use_age=False`` memberi model pembanding tanpa usia.\n    ``basis`` = basis usia yang sudah di-fit (dipakai bootstrap agar knot spline tidak berpindah)."""\n    device = device or ("cuda" if torch.cuda.is_available() else "cpu")\n    y = torch.tensor(df["helpful_vote"].to_numpy(np.float32), device=device)\n    pid_codes, prods = pd.factorize(df["parent_asin"])\n    pid = torch.tensor(pid_codes, device=device, dtype=torch.long)\n    C_np, cov_stats = _cov_matrix(df, None)\n    C = torch.tensor(C_np, device=device)\n    if use_age and basis is None:\n        basis = AgeBasis(basis_kind).fit(df["log_age"].to_numpy())\n    basis = basis if use_age else None\n    B = torch.tensor(basis.transform(df["log_age"].to_numpy()), device=device) if use_age else None\n\n    # inisialisasi efek produk dari rata-rata suara produk\n    pm = df.groupby(pid_codes)["helpful_vote"].mean().to_numpy()\n    gamma = torch.nn.Parameter(torch.tensor(np.log(pm + 0.1), dtype=torch.float32, device=device))\n    beta = torch.nn.Parameter(torch.zeros(C.shape[1], device=device))\n    w = torch.nn.Parameter(torch.zeros(B.shape[1], device=device)) if use_age else None\n    log_kappa = torch.nn.Parameter(torch.zeros((), device=device))\n    params = [gamma, beta, log_kappa] + ([w] if use_age else [])\n    opt = torch.optim.Adam(params, lr=lr)\n    n = len(df)\n    for step in range(steps):\n        opt.zero_grad()\n        eta = gamma[pid] + C @ beta + (B @ w if use_age else 0.0)\n        mu = torch.exp(eta.clamp(-20, 15))\n        ll = nb_loglik(y, mu, log_kappa).sum()\n        loss = -ll / n + l2_gamma * (gamma ** 2).mean()\n        loss.backward()\n        opt.step()\n        if verbose and step % 100 == 0:\n            print(step, float(-ll / n), float(torch.exp(log_kappa)))\n    with torch.no_grad():\n        eta = gamma[pid] + C @ beta + (B @ w if use_age else 0.0)\n        mu = torch.exp(eta.clamp(-20, 15))\n        ll = float(nb_loglik(y, mu, log_kappa).sum())\n        kappa = float(torch.exp(log_kappa))\n        dev = nb_deviance(df["helpful_vote"].to_numpy(), mu.cpu().numpy(), kappa)\n    return FitResult(basis=basis, w=(w.detach().cpu().numpy().astype(np.float64) if use_age else np.zeros(0)),\n                     beta=beta.detach().cpu().numpy().astype(np.float64), cov_stats=cov_stats, kappa=kappa,\n                     loglik=ll, deviance=dev, gamma=dict(zip(prods, gamma.detach().cpu().numpy())))\n\n\ndef heldout_fit(fit: FitResult, df: pd.DataFrame, steps: int = 300, lr: float = 0.05,\n                device: str | None = None) -> dict:\n    """Log-likelihood dan deviance pada produk baru (validasi). f, beta, dan kappa dibekukan, hanya\n    efek produk baru yang diestimasi karena merupakan parameter pengganggu."""\n    device = device or ("cuda" if torch.cuda.is_available() else "cpu")\n    y = torch.tensor(df["helpful_vote"].to_numpy(np.float32), device=device)\n    pid_codes, _ = pd.factorize(df["parent_asin"])\n    pid = torch.tensor(pid_codes, device=device, dtype=torch.long)\n    C_np, _ = _cov_matrix(df, fit.cov_stats)\n    fixed = torch.tensor(C_np @ fit.beta.astype(np.float32) + fit.f(df["log_age"].to_numpy()).astype(np.float32),\n                         device=device)\n    pm = df.groupby(pid_codes)["helpful_vote"].mean().to_numpy()\n    gamma = torch.nn.Parameter(torch.tensor(np.log(pm + 0.1), dtype=torch.float32, device=device))\n    lk = torch.tensor(np.log(fit.kappa), dtype=torch.float32, device=device)\n    opt = torch.optim.Adam([gamma], lr=lr)\n    for _ in range(steps):\n        opt.zero_grad()\n        mu = torch.exp((gamma[pid] + fixed).clamp(-20, 15))\n        (-nb_loglik(y, mu, lk).mean()).backward()\n        opt.step()\n    with torch.no_grad():\n        mu_t = torch.exp((gamma[pid] + fixed).clamp(-20, 15))\n        ll = float(nb_loglik(y, mu_t, lk).sum())\n        mu = mu_t.cpu().numpy()\n    return {"loglik": ll, "deviance": nb_deviance(df["helpful_vote"].to_numpy(), mu, fit.kappa)}\n\n\ndef select_basis(train: pd.DataFrame, val: pd.DataFrame, grid, **kw) -> tuple[object, pd.DataFrame]:\n    """Pilih bentuk fungsi usia (linear atau spline) dengan log-likelihood data validasi terbesar.\n    Bukan deviance, karena setiap basis punya kappa sendiri dan deviance NB dengan kappa berbeda\n    tidak bisa dibandingkan (lihat ``age_effect_stats``). Deviance tetap dicatat sebagai informasi."""\n    rows = []\n    for kind in grid:\n        fit = fit_age_model(train, basis_kind=kind, **kw)\n        ho = heldout_fit(fit, val)\n        if not np.isfinite(ho["loglik"]):\n            raise ValueError(f"log-likelihood validasi basis {kind} tidak valid: {ho[\'loglik\']}")\n        rows.append({"basis": str(kind), "train_loglik": fit.loglik, "val_loglik": ho["loglik"],\n                     "train_deviance": fit.deviance, "val_deviance": ho["deviance"], "kappa": fit.kappa})\n    tab = pd.DataFrame(rows)\n    best = tab.loc[tab["val_loglik"].idxmax(), "basis"]\n    best = "linear" if best == "linear" else int(best)\n    return best, tab\n\n\ndef age_effect_stats(loglik_full: float, loglik_noage: float, df: int) -> dict:\n    """Kontribusi komponen usia untuk RM3, dihitung dari log-likelihood saja.\n\n    Deviance NB bergantung pada dispersi kappa, dan model dengan dan tanpa usia masing-masing punya\n    kappa sendiri. Selisih kedua deviance itu karena itu tidak bermakna (bisa negatif walaupun usia\n    jelas memperbaiki model). Ukuran yang dipakai adalah uji rasio likelihood dan pseudo-R2 McFadden\n    terhadap model tanpa usia, ``1 - LL_full / LL_noage`` (bagian log-likelihood yang diperbaiki usia)."""\n    from scipy.stats import chi2\n    if not (np.isfinite(loglik_full) and np.isfinite(loglik_noage) and loglik_noage < 0):\n        raise ValueError(f"log-likelihood tidak valid: full={loglik_full}, tanpa usia={loglik_noage}")\n    lr_stat = 2.0 * (loglik_full - loglik_noage)\n    return {"lr_stat": float(lr_stat), "df": int(df), "p_value_lr": float(chi2.sf(lr_stat, df)),\n            "loglik_gain_share": float(1.0 - loglik_full / loglik_noage)}\n\n\ndef age_effect_summary(full: FitResult, noage: FitResult) -> dict:\n    """Ringkasan untuk RM3. ``deviance_full`` dan ``deviance_noage`` masing-masing dihitung dengan\n    kappa modelnya sendiri, jadi JANGAN dibandingkan langsung (lihat ``age_effect_stats``)."""\n    return {\n        "loglik_full": full.loglik, "loglik_noage": noage.loglik,\n        **age_effect_stats(full.loglik, noage.loglik, len(full.w)),\n        "deviance_full": full.deviance, "deviance_noage": noage.deviance,\n        "kappa_full": full.kappa,\n    }\n\n\ndef age_curve(fit: FitResult, ages_days: np.ndarray, ref_days: float = 30.0) -> np.ndarray:\n    """Efek multiplikatif usia terhadap rata-rata suara, relatif terhadap usia acuan.\n    Nilai 2,0 berarti rata-rata suara yang diharapkan dua kali lipat dibanding ulasan berusia ``ref_days``."""\n    la = np.log1p(np.asarray(ages_days, dtype=np.float64))\n    return np.exp(fit.f(la) - fit.f(np.log1p(np.array([ref_days])))[0])\n\n\ndef bootstrap_age_curve(train: pd.DataFrame, full: FitResult, ages_days, n_boot: int, seed: int,\n                        **kw) -> np.ndarray:\n    """Kurva usia hasil bootstrap pada tingkat produk, untuk IK kurva ``full``.\n\n    Setiap ulangan mengambil ulang produk (dengan pengembalian) dari SELURUH ``train``, yaitu data yang\n    sama dengan yang dipakai ``full``, dan memakai basis usia ``full`` (knot spline tetap). Dengan begitu\n    sebaran bootstrap berpusat pada estimasi yang sama dengan kurva yang dilaporkan."""\n    if full.basis is None:\n        raise ValueError("bootstrap_age_curve butuh model dengan komponen usia.")\n    rng = np.random.default_rng(seed)\n    base = train[["parent_asin", "helpful_vote", "log_age", *COVARIATES]]\n    codes, prods = pd.factorize(base["parent_asin"])\n    order = np.argsort(codes, kind="stable")\n    base = base.iloc[order].reset_index(drop=True)\n    counts = np.bincount(codes)\n    starts = np.r_[0, np.cumsum(counts)[:-1]]\n    curves = []\n    for _ in range(n_boot):\n        pick = rng.integers(0, len(prods), size=len(prods))\n        idx = np.concatenate([np.arange(starts[p], starts[p] + counts[p]) for p in pick])\n        sample = base.iloc[idx].copy()\n        # produk yang terambil lebih dari sekali dianggap produk berbeda\n        sample["parent_asin"] = np.repeat(np.arange(len(pick)), counts[pick])\n        fit = fit_age_model(sample, basis=full.basis, **kw)\n        curves.append(age_curve(fit, ages_days))\n    return np.vstack(curves)\n\n\ndef age_curve_table(full: FitResult, boots: np.ndarray, ages_days) -> pd.DataFrame:\n    """Tabel kurva usia (``tables/age_kurva.csv``): rasio dari ``full`` dan IK 95% persentil bootstrap.\n    Kolom ``di_luar_ik`` menandai titik yang estimasinya keluar dari IK-nya sendiri, dan jumlahnya\n    dicetak, karena hal itu menandakan bootstrap tidak sejalan dengan estimasi."""\n    ages = np.asarray(ages_days, dtype=np.float64)\n    ratio = age_curve(full, ages)\n    lo, hi = np.quantile(boots, [0.025, 0.975], axis=0)\n    tab = pd.DataFrame({"age_days": ages, "ratio": ratio, "lo": lo, "hi": hi})\n    tab["di_luar_ik"] = (ratio < lo) | (ratio > hi)\n    n_out = int(tab["di_luar_ik"].sum())\n    print(f"IK kurva usia dari {len(boots)} ulangan bootstrap. Titik di luar IK-nya sendiri: {n_out} dari {len(tab)}"\n          + ("" if n_out == 0 else ". PERIKSA sebelum dilaporkan."))\n    return tab\n\n\n# ---------------------------------------------------------------- target terkoreksi usia\ndef oof_age_component(train: pd.DataFrame, basis_kind, n_folds: int, seed: int, **kw) -> np.ndarray:\n    """f-hat out-of-fold untuk data latih. Fold dikelompokkan per produk."""\n    rng = np.random.default_rng(seed)\n    prods = train["parent_asin"].unique()\n    fold_of = dict(zip(prods, rng.integers(0, n_folds, size=len(prods))))\n    fold = train["parent_asin"].map(fold_of).to_numpy()\n    out = np.full(len(train), np.nan)\n    for k in range(n_folds):\n        tr = train[fold != k]\n        fit = fit_age_model(tr, basis_kind=basis_kind, **kw)\n        idx = np.where(fold == k)[0]\n        out[idx] = fit.f(train["log_age"].to_numpy()[idx])\n    if not np.isfinite(out).all():\n        raise ValueError(f"{(~np.isfinite(out)).sum()} komponen usia out-of-fold kosong/NaN "\n                         "(ada fold tanpa produk?). Target T3 tidak bisa dibentuk.")\n    return out\n\n\ndef target_t3(votes, f_hat: np.ndarray, f_bar: float) -> np.ndarray:\n    """Persamaan (2.18)."""\n    return target_t1(votes) - (np.asarray(f_hat) - f_bar)\n\n\n# ---------------------------------------------------------------- tingkat relevansi\ndef fit_grade_thresholds(y_train_t1: np.ndarray, y_train_target: np.ndarray | None = None) -> np.ndarray:\n    """Persamaan (3.3).\n\n    T1 dan T2: q1 = 0, sedangkan q2..q4 adalah kuartil target positif pada data latih.\n    T3: ambang dipilih sebagai kuantil target T3 pada data latih sehingga proporsi setiap tingkat\n    sama dengan proporsi tingkat T1. Dengan begitu kedua protokol memiliki sebaran tingkat yang setara.\n    """\n    y1 = np.asarray(y_train_t1, dtype=np.float64)\n    pos = y1[y1 > 0]\n    q_t1 = np.concatenate([[0.0], np.quantile(pos, [0.25, 0.5, 0.75])])\n    if y_train_target is None:\n        return q_t1\n    g1 = grades(y1, q_t1)\n    shares = np.array([(g1 < m).mean() for m in range(1, 5)])\n    return np.quantile(np.asarray(y_train_target, dtype=np.float64), shares)\n\n\ndef grades(y: np.ndarray, q: np.ndarray) -> np.ndarray:\n    y = np.asarray(y, dtype=np.float64)\n    return (y[:, None] > np.asarray(q)[None, :]).sum(axis=1).astype(np.int8)\n',
    'src/hr/train.py': '"""Pelatihan pemberi skor (Subbab 3.7) dengan early stopping berdasarkan NDCG@10 validasi."""\nfrom __future__ import annotations\n\nimport copy\nimport time\nfrom dataclasses import dataclass\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nfrom .metrics import macro, per_product_metrics\nfrom .models import Scorer, pointwise_mse, ranknet_loss\nfrom .pairs import PairSampler\n\n\n@dataclass\nclass EvalSet:\n    """Data evaluasi beserta label yang dibutuhkan metrik."""\n    X: np.ndarray\n    pid: np.ndarray\n    rid: np.ndarray\n    grade: np.ndarray\n    target: np.ndarray\n\n\ndef set_seed(seed: int) -> None:\n    import random\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n\n\n@torch.no_grad()\ndef predict(model: torch.nn.Module, X, device: str, batch: int = 65536) -> np.ndarray:\n    model.eval()\n    out = []\n    for i in range(0, len(X), batch):\n        xb = torch.as_tensor(np.asarray(X[i:i + batch]), device=device)\n        out.append(model(xb).float().cpu().numpy())\n    s = np.concatenate(out) if out else np.zeros(0)\n    if not np.isfinite(s).all():\n        raise FloatingPointError(f"{(~np.isfinite(s)).sum()} skor prediksi NaN/inf. Model divergen.")\n    return s\n\n\ndef evaluate_scores(es: EvalSet, scores: np.ndarray, map_rel: int = 2) -> tuple[dict, pd.DataFrame]:\n    pp = per_product_metrics(es.pid, scores, es.grade, es.target, es.rid, map_rel=map_rel)\n    return macro(pp), pp\n\n\ndef train_scorer(X_train: np.ndarray, y_train: np.ndarray, pid_train: np.ndarray, val: EvalSet, *,\n                 loss: str = "ranknet", hidden=(256, 64), dropout: float = 0.2, lr: float = 3e-4,\n                 weight_decay: float = 1e-5, max_epochs: int = 30, patience: int = 3,\n                 batch_pairs: int = 1024, batch_points: int = 1024, max_pairs: int = 100,\n                 delta: float = 0.0, log_age_train: np.ndarray | None = None,\n                 age_window: float | None = None, seed: int = 42, map_rel: int = 2,\n                 device: str | None = None, verbose: bool = True):\n    """Latih Scorer. Mengembalikan (model terbaik, riwayat per epoch, metrik validasi terbaik).\n\n    Riwayat berisi loss latih, loss validasi (pada pasangan validasi yang tetap), dan NDCG@10\n    validasi setiap epoch. Riwayat ini dipakai untuk grafik loss di Bab IV.\n    """\n    device = device or ("cuda" if torch.cuda.is_available() else "cpu")\n    set_seed(seed)\n    Xt = torch.as_tensor(np.ascontiguousarray(X_train), device=device)\n    model = Scorer(Xt.shape[1], hidden=hidden, dropout=dropout).to(device)\n    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)\n\n    sampler = PairSampler(pid_train, y_train, max_pairs, delta, log_age_train, age_window, seed=seed)\n    yt = torch.as_tensor(np.asarray(y_train, dtype=np.float32), device=device)\n    # pasangan validasi tetap untuk memantau loss validasi\n    vs = PairSampler(val.pid, val.target, max_pairs, delta, seed=seed + 999)\n    v_hi, v_lo = vs.sample()\n\n    best, best_state, bad, hist = -np.inf, None, 0, []\n    for ep in range(1, max_epochs + 1):\n        model.train()\n        t0 = time.time()\n        losses = []\n        if loss == "ranknet":\n            hi, lo = sampler.sample()\n            hi_t = torch.as_tensor(hi, device=device)\n            lo_t = torch.as_tensor(lo, device=device)\n            for i in range(0, len(hi_t), batch_pairs):\n                bh, bl = hi_t[i:i + batch_pairs], lo_t[i:i + batch_pairs]\n                l = ranknet_loss(model(Xt[bh]), model(Xt[bl]))\n                opt.zero_grad()\n                l.backward()\n                opt.step()\n                losses.append(l.item())\n            n_pairs = len(hi)\n            if n_pairs == 0:\n                raise ValueError("Tidak ada pasangan latih (semua target sama dalam setiap produk, atau "\n                                 "age_window terlalu sempit). Model tidak akan belajar apa pun.")\n        elif loss == "mse":\n            perm = torch.randperm(len(Xt), device=device)\n            for i in range(0, len(perm), batch_points):\n                b = perm[i:i + batch_points]\n                l = pointwise_mse(model(Xt[b]), yt[b])\n                opt.zero_grad()\n                l.backward()\n                opt.step()\n                losses.append(l.item())\n            n_pairs = 0\n        else:\n            raise ValueError(loss)\n\n        if not losses or not np.isfinite(losses).all():\n            raise FloatingPointError(f"Loss latih tidak valid pada epoch {ep} (kosong atau NaN/inf). "\n                                     "Coba learning rate lebih kecil.")\n        scores = predict(model, val.X, device)\n        m, _ = evaluate_scores(val, scores, map_rel)\n        with torch.no_grad():\n            model.eval()\n            sv = torch.as_tensor(scores, device=device)\n            vloss = float(ranknet_loss(sv[torch.as_tensor(v_hi, device=device)],\n                                       sv[torch.as_tensor(v_lo, device=device)])) if len(v_hi) else float("nan")\n        rec = {"epoch": ep, "train_loss": float(np.mean(losses)) if losses else float("nan"),\n               "val_ranknet_loss": vloss, "val_ndcg@10": m["ndcg@10"], "val_ndcg@5": m["ndcg@5"],\n               "val_spearman": m["spearman"], "val_map": m["map"], "pairs": n_pairs,\n               "seconds": time.time() - t0}\n        hist.append(rec)\n        if verbose:\n            print(f"epoch {ep:2d}  train {rec[\'train_loss\']:.4f}  val_loss {vloss:.4f}  "\n                  f"val NDCG@10 {m[\'ndcg@10\']:.4f}")\n        if m["ndcg@10"] > best + 1e-6:\n            best, best_state, bad = m["ndcg@10"], copy.deepcopy(model.state_dict()), 0\n        else:\n            bad += 1\n            if bad >= patience:\n                break\n    if best_state is None:\n        raise ValueError("NDCG@10 validasi NaN di setiap epoch: tidak ada produk validasi dengan IDCG > 0.")\n    model.load_state_dict(best_state)\n    del Xt\n    return model, pd.DataFrame(hist), best\n',
}
_VERSI = 'ebf37aa093df'
assert hashlib.sha256(json.dumps(_KODE, sort_keys=True).encode()).hexdigest()[:12] == _VERSI, "kode tertanam rusak"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB or os.environ.get("HR_EMBEDDED") == "1":
    ROOT = Path(os.environ.get("HR_KODE_DIR") or
                ("/content/hr_kode" if IN_COLAB else tempfile.mkdtemp(prefix="hr_kode_"))).resolve()
    # Saat sel ini dijalankan ulang, direktori kerja kernel masih di dalam ROOT (os.chdir di bawah).
    # Pindah keluar dulu sebelum ROOT dihapus, agar pip tidak berjalan di direktori yang sudah tiada.
    ROOT.parent.mkdir(parents=True, exist_ok=True)
    os.chdir(ROOT.parent)
    shutil.rmtree(ROOT, ignore_errors=True)            # buang kode sesi sebelumnya
    for rel, text in _KODE.items():
        (ROOT / rel).parent.mkdir(parents=True, exist_ok=True)
        (ROOT / rel).write_text(text, encoding="utf-8")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")],
                       capture_output=True, text=True, cwd=ROOT)
    if r.returncode != 0:
        raise RuntimeError("pip install gagal:\n" + (r.stderr or r.stdout)[-3000:])
    os.chdir(ROOT / "notebooks")
    sumber = "tertanam"
else:                                                  # di luar Colab: pakai kode repo tempat notebook berada
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    sumber = "repo lokal"
for _m in [m for m in sys.modules if m in ("hr", "settings") or m.startswith("hr.")]:
    del sys.modules[_m]                                # sel dijalankan ulang: muat kode baru, bukan versi lama
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "notebooks")]
del _KODE
print("kode:", ROOT, f"({sumber}, versi {_VERSI})")

In [ ]:
#@title 2 · Run yang direkonsiliasi { display-mode: "form" }
kategori = "Video_Games"  #@param ["All_Beauty", "Amazon_Fashion", "Appliances", "Arts_Crafts_and_Sewing", "Automotive", "Baby_Products", "Beauty_and_Personal_Care", "Books", "CDs_and_Vinyl", "Cell_Phones_and_Accessories", "Clothing_Shoes_and_Jewelry", "Digital_Music", "Electronics", "Gift_Cards", "Grocery_and_Gourmet_Food", "Handmade_Products", "Health_and_Household", "Health_and_Personal_Care", "Home_and_Kitchen", "Industrial_and_Scientific", "Kindle_Store", "Magazine_Subscriptions", "Movies_and_TV", "Musical_Instruments", "Office_Products", "Patio_Lawn_and_Garden", "Pet_Supplies", "Software", "Sports_and_Outdoors", "Subscription_Boxes", "Tools_and_Home_Improvement", "Toys_and_Games", "Unknown", "Video_Games"] {allow-input: true}
folder_drive = "/content/drive/MyDrive/dl-paper"  #@param {type:"string"}

import json, shutil
import numpy as np, pandas as pd
pd.set_option("display.width", 160)
import settings
from settings import SMOKE, finish
cfg = settings.configure_saved(kategori, drive_root=folder_drive)   # pengaturan diambil dari config.json run lama
print("kategori:", cfg.category, "| output:", cfg.out_dir, "| SMOKE:", SMOKE)
from hr import targets, features, plots, experiment as X, report as R
from hr.train import evaluate_scores
from hr.config import BASELINE_CODES
T = lambda name: cfg.path("tables", name)
F = lambda name: cfg.path("figures", name)
diubah = []


def cadangkan(p):
    """Salin berkas sekali sebelum diubah. Cadangan yang sudah ada tidak ditimpa (isinya versi asli)."""
    b = p.with_name(p.stem + ".sebelum_rekonsiliasi" + p.suffix)
    if p.exists() and not b.exists():
        shutil.copy2(p, b)
    diubah.append(p.name)

In [ ]:
#@title 3 · age_model.json: ganti deviance_reduction_share dengan loglik_gain_share
p = cfg.path("age_model.json")
summ = json.loads(p.read_text())
stats = targets.age_effect_stats(summ["loglik_full"], summ["loglik_noage"], summ["df"])
if not np.isclose(stats["lr_stat"], summ["lr_stat"], rtol=1e-9):
    raise RuntimeError(f"lr_stat tersimpan ({summ['lr_stat']}) tidak cocok dengan log-likelihood tersimpan. "
                       "age_model.json tidak konsisten. Jalankan ulang tahap 03.")
# urutan kunci sama dengan keluaran tahap 03 versi baru
fixed = {"loglik_full": summ["loglik_full"], "loglik_noage": summ["loglik_noage"], **stats,
         "deviance_full": summ["deviance_full"], "deviance_noage": summ["deviance_noage"],
         "kappa_full": summ["kappa_full"], "basis": summ["basis"], "beta": summ["beta"]}
if fixed != summ:
    cadangkan(p)
    p.write_text(json.dumps(fixed, indent=2, default=str))
    print("diperbarui:", p)
else:
    print("sudah sesuai, tidak diubah:", p)
pd.Series(fixed)

In [ ]:
#@title 4 · Pemilihan basis usia dengan log-likelihood validasi
largest = list(cfg.scales)[-1]
cols = sorted({"split", f"in_{largest}", "parent_asin", "helpful_vote", "log_age", *targets.COVARIATES})
df = pd.read_parquet(cfg.path("reviews_split.parquet"), columns=cols)
tr = df[(df.split == "train") & df[f"in_{largest}"]].reset_index(drop=True)   # sama dengan tahap 03
va = df[df.split == "val"].reset_index(drop=True)
del df
best_kind, basis_tab = targets.select_basis(tr, va, cfg.age_basis_grid)
del va                                          # tr dipakai lagi di sel berikutnya
p = cfg.path("tables", "age_pemilihan_basis.csv")
cadangkan(p)
basis_tab.to_csv(p, index=False)
display(basis_tab)
lama = json.loads(cfg.path("age_model.json").read_text())["basis"]
if str(best_kind) != str(lama):
    raise RuntimeError(
        f"Basis terpilih berubah: {lama} (run lama, dipilih dari deviance) menjadi {best_kind} (log-likelihood "
        "validasi). Target T3 harus dibentuk ulang: hapus folder runs/ yang namanya berisi '_T3_' lalu jalankan "
        "00_pipeline_lengkap.ipynb dengan pengaturan yang sama dan mulai_dari_tahap = 3. Embedding dan run T1/T2 "
        "dipakai ulang. Sel berikut tidak dijalankan.")
print(f"basis terpilih tetap {best_kind}: target T3 dan semua hasil yang bergantung padanya tetap berlaku.")

In [ ]:
#@title 5 · IK kurva usia: bootstrap pada seluruh data latih dengan knot tetap
am = json.loads(cfg.path("age_model.json").read_text())
full = targets.fit_age_model(tr, basis_kind=best_kind)          # model yang sama dengan tahap 03
if not np.isclose(full.loglik, am["loglik_full"], rtol=1e-4):
    raise RuntimeError(f"Fit ulang model usia memberi log-likelihood {full.loglik}, bukan {am['loglik_full']} "
                       "seperti di age_model.json. Data latih berubah sejak run lama. Jalankan ulang dari tahap 03.")
p = T("age_kurva.csv")
old = pd.read_csv(p)
ages = old["age_days"].to_numpy()                                # grid usia yang sama dengan run lama
if not np.allclose(targets.age_curve(full, ages), old["ratio"], rtol=1e-3):
    raise RuntimeError("Kurva usia hasil fit ulang berbeda dari age_kurva.csv. Jalankan ulang dari tahap 03.")
if "di_luar_ik" in old.columns:
    # sudah dibuat dengan bootstrap versi baru: jangan diulang, karena hasil GPU bisa sedikit berbeda
    curve_tab = old
    print("sudah sesuai, tidak diubah:", p)
else:
    boots = targets.bootstrap_age_curve(tr, full, ages, n_boot=cfg.age_bootstrap, seed=cfg.seed)
    curve_tab = targets.age_curve_table(full, boots, ages)
    cadangkan(p)
    curve_tab.to_csv(p, index=False)
    ax = plots.age_curve(curve_tab["age_days"], curve_tab["ratio"], curve_tab["lo"], curve_tab["hi"])
    cadangkan(F("age_kurva.png"))
    plots.save(ax.figure, F("age_kurva.png"))
    del boots
del tr, full
curve_tab.round(4)

In [ ]:
#@title 6 · Skor validasi B0 dan tabel validasi
MS, TG = cfg.main_scale, cfg.main_target
run = X.run_id("B0", MS, TG, 0)
rp = cfg.out_dir / "runs" / run / "result.json"
meta = json.loads(rp.read_text())
b = X.load_bundle(cfg)
va = b.rows("val")
score = features.score_b0(va)
saved = X.load_pred(cfg, run, "val")
# skor dihitung ulang dengan kode yang sama, dan harus identik dengan prediksi B0 yang tersimpan
if not (np.array_equal(saved["rid"].to_numpy(), va["rid"].to_numpy())
        and np.allclose(saved["score"].to_numpy(), score)):
    raise RuntimeError(f"Skor B0 yang dihitung ulang berbeda dari {run}/pred_val.parquet. Data validasi berubah "
                       "sejak run lama. Jalankan ulang dari tahap 05.")
m = evaluate_scores(b.eval_set(va, np.zeros((len(va), 1)), TG), score, cfg.map_relevant_grade)[0]
if meta.get("best_val_ndcg@10") is None:
    cadangkan(rp)
    rp.write_text(json.dumps({**meta, "best_val_ndcg@10": m["ndcg@10"]}, indent=2, default=str))
    print("diperbarui:", rp)
elif not np.isclose(meta["best_val_ndcg@10"], m["ndcg@10"]):
    raise RuntimeError(f"best_val_ndcg@10 B0 tersimpan {meta['best_val_ndcg@10']} berbeda dari {m['ndcg@10']}.")
else:
    print("sudah sesuai, tidak diubah:", rp)
del va                                          # b dipakai lagi di sel berikutnya

for name, tab, kw in (("validasi_baseline.csv", X.validation_table(cfg, MS, TG, BASELINE_CODES), {}),
                      ("validasi_semua.csv", X.validation_table(cfg), {"index": False})):
    p = cfg.path("tables", name)
    cadangkan(p)
    tab.to_csv(p, **kw)
    print(name); display(tab)

In [ ]:
#@title 7 · Urutan acak R0, tabel ukuran produk, dan grafik
groups = R.group_runs(X.results_table(cfg))
R.check_complete(cfg, groups)
te = b.rows("test")
N_PERM = R.R0_PERMUTATIONS if not SMOKE else 3                  # sama dengan notebook 07
pp_r0 = R.random_per_product(b, TG, n_perm=N_PERM, seed=cfg.seed)
r0_tab = R.random_table(pp_r0, N_PERM, cfg.eval_bootstrap)
cadangkan(T("uji_acak.csv"))
r0_tab.to_csv(T("uji_acak.csv"), index=False)
display(r0_tab[["code", "seeds", "ndcg@10", "ndcg@10_lo", "ndcg@10_hi", "ndcg@5", "spearman", "map"]].round(4))

pp_u = R.eval_config(b, groups[("U", MS, TG)])
size_tab = R.size_table(pp_u, pp_r0, te)
p = T("uji_per_ukuran_produk.csv")
old = pd.read_csv(p).set_index("kelompok")
cols = ["ndcg@10", "spearman", "ap"]
if not np.allclose(old[cols].to_numpy(), size_tab[cols].to_numpy(), atol=1e-4):
    raise RuntimeError("Metrik model U per ukuran produk berbeda dari tabel lama. Prediksi atau data uji berubah.")
cadangkan(p)
size_tab.to_csv(p)
display(size_tab)
del te, pp_r0, pp_u                             # b dan groups dipakai lagi di sel berikutnya

for name, make in (("uji_baseline.png", lambda: plots.baseline_figure(pd.read_csv(T("uji_baseline.csv")),
                                                                    pd.read_csv(T("uji_acak.csv"))).figure),
                   ("usia.png", lambda: plots.age_figure(pd.read_csv(T("age_kurva.csv")),
                                                         pd.read_csv(T("uji_per_kelompok_usia.csv")))),
                   ("uji_ukuran_produk.png", lambda: plots.size_figure(pd.read_csv(T("uji_per_ukuran_produk.csv"))).figure)):
    cadangkan(F(name))
    plots.save(make(), F(name))
    print("digambar ulang:", F(name))

In [ ]:
#@title 8 · Konsistensi per seed pada perbandingan berpasangan (data uji)
keys = [(c, MS, TG) for c in BASELINE_CODES]
others = [k for k in keys if k[0] != "U"]
_, pps = R.comparison_table(b, groups, keys, TG, cfg.eval_bootstrap)
pair_tab = R.paired_table(pps, ("U", MS, TG), others, "ndcg@10", cfg.eval_bootstrap)
p = T("uji_selisih_berpasangan.csv")
old = pd.read_csv(p)
core = ["diff", "lo", "hi", "p", "p_holm"]
if (old["vs"].tolist() != pair_tab["vs"].tolist()
        or not np.allclose(old[core].to_numpy(), pair_tab[core].to_numpy(), rtol=0, atol=1e-9)
        or old["lebih_baik"].astype(bool).tolist() != pair_tab["lebih_baik"].tolist()):
    raise RuntimeError("Selisih berpasangan yang dihitung ulang berbeda dari uji_selisih_berpasangan.csv. "
                       "Prediksi atau data uji berubah sejak run lama. Jalankan ulang dari tahap 07.")
pair_tab = R.add_seed_consistency(pair_tab, b, groups, ("U", MS, TG), others, "ndcg@10", TG)
new_cols = ["selisih_per_seed", "seed_searah"]
if all(c in old.columns for c in new_cols) and all(old[c].tolist() == pair_tab[c].tolist() for c in new_cols):
    print("sudah sesuai, tidak diubah:", p)
else:
    cadangkan(p)
    pair_tab.to_csv(p, index=False)
    print("diperbarui:", p)
del b, pps
pair_tab[["vs", "diff", "lo", "hi", "p_holm", "lebih_baik", "selisih_per_seed", "seed_searah",
          "n_pasangan_seed", "searah_semua_seed"]].round(4)

In [ ]:
#@title 9 · Selesai
print("berkas yang ditulis ulang (cadangan *.sebelum_rekonsiliasi.* di folder yang sama):", sorted(set(diubah)))
finish()